# Your 3D Universe Explorer
### Solar System → moons → Milky Way → stars → galaxies
**Prepared by Hassam · 10 September 2026 · Python / Jupyter / Plotly**

Open this single notebook in your own Jupyter Notebook or JupyterLab. Run the numbered code cells in order with **Shift + Enter**. Drag a 3D plot to rotate, scroll to zoom, hover for values, and use Play or the time slider where shown. Double-click a legend item to isolate it.

**Start here:** run Cell 01 once to install packages. If prompted, restart the kernel, then run from Cell 02. The saved notebook contains demonstration outputs. Trust the downloaded notebook if Jupyter suppresses its interactive outputs, then rerun the plot cells.

The default run uses embedded data and models; it does not require NASA/ESA servers. Optional online sections add JPL ephemerides, Gaia stars and NASA's known exoplanets. You can enable them individually in Cell 03 and rerun their cells.

### What you can explore
| View | What it contains | What its movement means |
|---|---|---|
| Inner and full Solar System | Eight planetary orbits; Earth–Moon barycentre in the approximate mode | Time-dependent JPL approximate elements; numerical speed |
| Every planet's moon system | All entries in the embedded JPL satellite table, including Pluto as an extra dwarf planet | Explicit circular, coplanar teaching model; individual mean orbital radii and periods |
| Accurate selected moon/planet view | Optional JPL Horizons vectors | Geometric position and velocity at a specified TDB epoch |
| Milky Way | Synthetic disc, centre, Sun and Solar orbit | An explicitly assumed circular rotation model |
| Your sky | Galactic plane and Galactic centre from Melbourne or your coordinates | Geometric altitude/azimuth at your chosen UTC time |
| Nearby stars | Optional measured Gaia DR3 sample | Measured 3D velocities, short straight-line extrapolation |
| Exoplanets | Optional all NASA archive entries with valid host positions/distances | Host locations, not unknown planetary orbital orientations |
| Nearby galaxies | Selected real galaxies, rounded reference values | Radial-velocity component only; full motion is not inferred |
| Expansion | Synthetic points under an assumed Hubble law | A teaching model, not a galaxy survey |

### Scientific boundaries
We do not know the positions and velocities of every planet, moon, star or galaxy. “All moons” here means **all entries available in the saved JPL table**, not every undiscovered or uncatalogued moon. “All exoplanets” means all returned archive entries passing the stated coordinate filter. There is no absolute speed through the Universe: every speed needs a reference frame. Separate scales are essential: a moon becomes invisible on a galaxy-scale axis. Object marker sizes are exaggerated; spatial axes are linear and use equal physical units unless explicitly marked otherwise. The Milky Way and expansion particles are **synthetic**, not observed stars/galaxies.

## Cell 01 · Install packages — first run only

Python 3.11 or 3.12 is recommended. This installs into the active notebook kernel. Internet access is needed for installation. If imports fail after installation, use Kernel → Restart and continue at Cell 02.

## Cell 02 · Imports and plotting setup

In [2]:
import hashlib
import io
import json
from pathlib import Path
import warnings
from datetime import datetime, timezone

import numpy as np
from astropy import units as u
from astropy.coordinates import (
    AltAz,
    CartesianDifferential,
    EarthLocation,
    Galactic,
    Galactocentric,
    SkyCoord,
)
from astropy.time import Time
from astropy.utils import iers
from IPython.display import display
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests

# Avoid unexpected Earth-orientation network requests during offline execution
iers.conf.auto_download = False
pio.templates.default = 'plotly_dark'
pio.renderers.default = 'notebook'

AU_KM = u.au.to(u.km)
DAY_S = 86400.0
KM_S_TO_PC_YR = (1 * u.km / u.s).to_value(u.pc / u.yr)
FIGURES = {}
CACHE = Path('universe_cache')
CACHE.mkdir(exist_ok=True)

print(
    'Ready. Distances and velocities will always have explicit units and'
    ' frames.'
)

Ready. Distances and velocities will always have explicit units and frames.


## Cell 03 · Your controls

Edit these values when you want a different date, location or moon system. After changing a control, rerun the relevant later cells. Online sections cache downloaded data beside this notebook; catalogue refresh is explicit.

In [3]:
# UTC for the sky view; the ephemeris code converts this instant to TDB.
EPOCH_UTC = '2026-09-10 12:00:00'
LATITUDE_DEG, LONGITUDE_DEG, HEIGHT_M = -37.8136, 144.9631, 30  # Melbourne

# Optional online modules. False keeps the default run independent of remote services.
RUN_HORIZONS = False
RUN_GAIA = False
RUN_EXOPLANETS = False
REFRESH_MOON_CATALOGUE = False

MOON_PLANET = 'Jupiter'  # Earth, Mars, Jupiter, Saturn, Uranus, Neptune, Pluto
MOON_DISPLAY = 'major'  # 'major' or 'all' (all may produce a large plot)
HORIZONS_PARENT = 'Jupiter'
HORIZONS_SCOPE = 'major'  # 'all' requests EVERY catalogued moon of this parent sequentially
HORIZONS_DAYS = 8.0
HORIZONS_SAMPLES = 81  # reduce DAYS / increase SAMPLES to resolve very fast moons
GAIA_MAX_STARS = 1500
GAIA_RADIUS_PC = 100
STAR_YEARS = 10000  # short, constant-velocity extrapolation, not integrated Galactic orbits
EXPORT_HTML = False  # True writes standalone rotatable plots beside this notebook
EPOCH_TDB = Time(EPOCH_UTC, scale='utc').tdb.jd
print('UTC instant:', EPOCH_UTC, '| TDB Julian date:', EPOCH_TDB)

UTC instant: 2026-09-10 12:00:00 | TDB Julian date: 2461294.0008007237


## Cell 04 · Shared 3D plotting and animation functions

In [4]:
def style3d(fig, title, unit, extent=None):
    scene = dict(xaxis_title=f'X ({unit})', yaxis_title=f'Y ({unit})',
                 zaxis_title=f'Z ({unit})', aspectmode='data',
                 bgcolor='#080d1d', camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)))
    if extent is not None:
        for axis in ['xaxis', 'yaxis', 'zaxis']:
            scene[axis] = dict(range=[-extent, extent], autorange=False)
        scene['aspectmode'] = 'cube'
    fig.update_layout(title=title, scene=scene, height=700,
                      margin=dict(l=0, r=0, t=90, b=80), uirevision=title,
                      legend=dict(itemsizing='constant'))
    return fig

def show(fig, key):
    FIGURES[key] = fig
    fig.show(config={'scrollZoom': True, 'displaylogo': False})

def animate(fig, frames, labels, milliseconds=70):
    fig.frames = frames
    fig.update_layout(
        updatemenus=[dict(type='buttons', direction='left', x=0, y=-0.03,
            buttons=[dict(label='▶ Play', method='animate', args=[None,
                dict(frame=dict(duration=milliseconds, redraw=True),
                     transition=dict(duration=0), fromcurrent=True, mode='immediate')]),
                     dict(label='Pause', method='animate', args=[[None],
                dict(frame=dict(duration=0, redraw=False), mode='immediate')])])],
        sliders=[dict(active=0, x=0.22, len=0.75, y=-0.02,
            currentvalue=dict(prefix='Time: '), steps=[
                dict(method='animate', label=str(label), args=[[frame.name],
                     dict(mode='immediate', frame=dict(duration=0, redraw=True),
                          transition=dict(duration=0))])
                for frame, label in zip(frames, labels)])])
    return fig

def line3d(xyz, name, color=None, width=2, **kwargs):
    return go.Scatter3d(x=xyz[:,0], y=xyz[:,1], z=xyz[:,2], mode='lines',
                        name=name, line=dict(color=color, width=width), **kwargs)

def velocity_arrow(fig, origin, vector, label, color='cyan', length=1):
    origin, vector = np.asarray(origin,float), np.asarray(vector,float)
    direction = vector / np.linalg.norm(vector)
    tip = origin + length*direction
    fig.add_trace(line3d(np.array([origin,tip]), label, color, 5))
    fig.add_trace(go.Cone(x=[tip[0]], y=[tip[1]], z=[tip[2]],
        u=[direction[0]], v=[direction[1]], w=[direction[2]],
        sizemode='absolute', sizeref=length*0.13, anchor='tip',
        colorscale=[[0,color],[1,color]], showscale=False, showlegend=False,
        hovertext=label, hoverinfo='text'))
print('Plot tools loaded.')

Plot tools loaded.


## Cell 05 · JPL approximate planetary elements

These are JPL’s 1800–2050 approximate elements and rates. The Earth entry is the Earth–Moon barycentre. They are not Horizons precision; speeds in this first table are circular-orbit comparisons. Source: [JPL approximate positions](https://ssd.jpl.nasa.gov/planets/approx_pos.html).

In [5]:
PLANET_NAMES = ['Mercury', 'Venus', 'Earth–Moon barycentre', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune']
ELEMENTS_J2000 = np.array([[0.38709927, 0.20563593, 7.00497902, 252.2503235, 77.45779628, 48.33076593], [0.72333566, 0.00677672, 3.39467605, 181.9790995, 131.60246718, 76.67984255], [1.00000261, 0.01671123, -1.531e-05, 100.46457166, 102.93768193, 0.0], [1.52371034, 0.0933941, 1.84969142, -4.55343205, -23.94362959, 49.55953891], [5.202887, 0.04838624, 1.30439695, 34.39644051, 14.72847983, 100.47390909], [9.53667594, 0.05386179, 2.48599187, 49.95424423, 92.59887831, 113.66242448], [19.18916464, 0.04725744, 0.77263783, 313.23810451, 170.9542763, 74.01692503], [30.06992276, 0.00859048, 1.77004347, -55.12002969, 44.96476227, 131.78422574]])
ELEMENT_RATES = np.array([[3.7e-07, 1.906e-05, -0.00594749, 149472.67411175, 0.16047689, -0.12534081], [3.9e-06, -4.107e-05, -0.0007889, 58517.81538729, 0.00268329, -0.27769418], [5.62e-06, -4.392e-05, -0.01294668, 35999.37244981, 0.32327364, 0.0], [1.847e-05, 7.882e-05, -0.00813131, 19140.30268499, 0.44441088, -0.29257343], [-0.00011607, -0.00013253, -0.00183714, 3034.74612775, 0.21252668, 0.20469106], [-0.0012506, -0.00050991, 0.00193609, 1222.49362201, -0.41897216, -0.28867794], [-0.00196176, -4.397e-05, -0.00242939, 428.48202785, 0.40805281, 0.04240589], [0.00026291, 5.105e-05, 0.00035372, 218.45945325, -0.32241464, -0.00508664]])
PLANET_COLOURS = ['#b1b1b1','#efc56d','#48c4ff','#f37950','#d6af84','#f0da9b','#72e3dc','#658dff']
# Columns: semimajor axis AU, eccentricity, inclination deg, mean longitude deg,
# longitude of perihelion deg, ascending node deg. Rates are per Julian century.
planet_table = pd.DataFrame(ELEMENTS_J2000, index=PLANET_NAMES,
    columns=['a_AU','eccentricity','inclination_deg','L_deg','perihelion_deg','node_deg'])
planet_table['rough_period_years'] = planet_table.a_AU**1.5
planet_table['circular_speed_km_s'] = 29.7847 / np.sqrt(planet_table.a_AU)
display(planet_table[['a_AU','eccentricity','inclination_deg','rough_period_years','circular_speed_km_s']].round(4))

,a_AU,eccentricity,inclination_deg,rough_period_years,circular_speed_km_s
Mercury,0.3871,0.2056,7.0050,0.2408,47.8721
Venus,0.7233,0.0068,3.3947,0.6152,35.0206
Earth–Moon barycentre,1.0000,0.0167,-0.0000,1.0000,29.7847
Mars,1.5237,0.0934,1.8497,1.8808,24.1291
Jupiter,5.2029,0.0484,1.3044,11.8677,13.0578
Saturn,9.5367,0.0539,2.4860,29.4507,9.6448
Uranus,19.1892,0.0473,0.7726,84.0590,6.7993
Neptune,30.0699,0.0086,1.7700,164.8916,5.4316


## Cell 06 · Solve Kepler’s equation and calculate positions

Kepler’s equation is $M=E-e\sin E$. The code solves it, rotates orbital coordinates into the J2000 ecliptic plane and derives velocity numerically. Reference origin: Sun. X points toward the J2000 equinox; Z toward ecliptic north.

In [6]:
def planet_xyz(jd):
    jd = np.atleast_1d(np.asarray(jd, float))
    if np.any((jd < 2378496.5) | (jd > 2469807.5)):
        raise ValueError('Keep this approximate model within 1800–2050; use Horizons outside it.')
    T = (jd-2451545.0)/36525
    el = ELEMENTS_J2000[None,:,:] + T[:,None,None]*ELEMENT_RATES[None,:,:]
    a,e,inc,L,peri,node = np.moveaxis(el,2,0)
    M = np.deg2rad((L-peri+180)%360-180)
    E = M.copy()
    for _ in range(14):
        E -= (E-e*np.sin(E)-M)/(1-e*np.cos(E))
    xp = a*(np.cos(E)-e); yp = a*np.sqrt(1-e*e)*np.sin(E)
    w, O, I = np.deg2rad(peri-node), np.deg2rad(node), np.deg2rad(inc)
    x = (np.cos(w)*np.cos(O)-np.sin(w)*np.sin(O)*np.cos(I))*xp + (-np.sin(w)*np.cos(O)-np.cos(w)*np.sin(O)*np.cos(I))*yp
    y = (np.cos(w)*np.sin(O)+np.sin(w)*np.cos(O)*np.cos(I))*xp + (-np.sin(w)*np.sin(O)+np.cos(w)*np.cos(O)*np.cos(I))*yp
    z = np.sin(w)*np.sin(I)*xp + np.cos(w)*np.sin(I)*yp
    return np.stack([x,y,z],axis=-1)  # time × planet × xyz, in AU

def planet_velocity(jd):
    h = 0.01 # centred finite difference, days
    return (planet_xyz(jd+h)[0]-planet_xyz(jd-h)[0])/(2*h)*AU_KM/DAY_S

p0=planet_xyz(EPOCH_TDB)[0]; v0=planet_velocity(EPOCH_TDB)
speed_table=pd.DataFrame({'body':PLANET_NAMES,'distance_from_Sun_AU':np.linalg.norm(p0,axis=1),
                          'speed_relative_to_Sun_km_s':np.linalg.norm(v0,axis=1)})
display(speed_table.round(4))

,body,distance_from_Sun_AU,speed_relative_to_Sun_km_s
0,Mercury,0.4282,43.0363
1,Venus,0.7282,34.7880
2,Earth–Moon barycentre,1.0070,29.5775
3,Mars,1.5322,23.9959
4,Jupiter,5.2979,12.8272
5,Saturn,9.4408,9.7432
6,Uranus,19.4394,6.7145
7,Neptune,29.8778,5.4699


## Cell 07 · Animate the inner Solar System

Play advances all planets by the same amount of physical time. Marker sizes are enlarged; distances are linear. Orbital paths are guides at the selected epoch.

In [7]:
def solar_animation(indices, days, key):
    times=np.linspace(0,days,121)
    xyz=planet_xyz(EPOCH_TDB+times)
    fig=go.Figure()
    for j in indices:
        # Schematic closed ellipse at the selected epoch; the animation uses evolving elements.
        el=ELEMENTS_J2000[j]+(EPOCH_TDB-2451545)/36525*ELEMENT_RATES[j]
        a,e,I,L,peri,O=el
        E=np.linspace(0,2*np.pi,240); xp=a*(np.cos(E)-e); yp=a*np.sqrt(1-e*e)*np.sin(E)
        w,O,I=np.deg2rad([peri-O,O,I])
        curve=np.column_stack([(np.cos(w)*np.cos(O)-np.sin(w)*np.sin(O)*np.cos(I))*xp+(-np.sin(w)*np.cos(O)-np.cos(w)*np.sin(O)*np.cos(I))*yp,
            (np.cos(w)*np.sin(O)+np.sin(w)*np.cos(O)*np.cos(I))*xp+(-np.sin(w)*np.sin(O)+np.cos(w)*np.cos(O)*np.cos(I))*yp,
            np.sin(w)*np.sin(I)*xp+np.cos(w)*np.sin(I)*yp])
        fig.add_trace(line3d(curve,PLANET_NAMES[j]+' orbit',PLANET_COLOURS[j],1,showlegend=False))
    fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',name='Sun',marker=dict(size=11,color='gold')))
    marker_index=len(fig.data)
    def marker(k):
        q=xyz[k,indices,:]
        speeds=np.linalg.norm(planet_velocity(EPOCH_TDB+times[k])[indices],axis=1)
        return go.Scatter3d(x=q[:,0],y=q[:,1],z=q[:,2],mode='markers+text',
            text=[PLANET_NAMES[j] for j in indices],textposition='top center',
            marker=dict(size=6,color=[PLANET_COLOURS[j] for j in indices]),
            customdata=speeds, hovertemplate='%{text}<br>Sun-relative speed: %{customdata:.2f} km/s<extra></extra>',name='Planets')
    fig.add_trace(marker(0))
    frames=[go.Frame(name=str(k),data=[marker(k)],traces=[marker_index]) for k in range(len(times))]
    extent=max(ELEMENTS_J2000[indices,0])*1.15
    style3d(fig,f'{key.replace("_"," ").title()} · approximate JPL model · days after {EPOCH_UTC} UTC','AU',extent)
    animate(fig,frames,[f'{t:.0f} d' for t in times]);show(fig,key)
solar_animation([0,1,2,3],365,'inner_solar_system')

## Cell 08 · Explore all eight planetary orbits

The outer planets move only partway around their orbits in two years. Inner planets are small at this scale. The coarser time steps can undersample Mercury; use the inner view to study it. This is orbital motion, not axial rotation.

In [8]:
solar_animation(list(range(8)),365*2,'full_solar_system')

## Cell 09 · Load the complete saved JPL moon catalogue

Mercury and Venus have no known natural moons. Pluto is included separately as a dwarf planet. The saved table covers 460 entries at preparation time. Mean elements describe orbital shape; [JPL explicitly recommends Horizons for accurate ephemerides](https://ssd.jpl.nasa.gov/sats/elem/).

In [9]:
MOON_CATALOGUE_JSON = '[{"planet":"Earth","moon":"Moon","id":"301","a_km":384400.0,"eccentricity":0.0554,"inclination_deg":5.16,"period_days":27.322,"reference_plane":"ecliptic"},{"planet":"Mars","moon":"Phobos","id":"401","a_km":9375.0,"eccentricity":0.015,"inclination_deg":1.1,"period_days":0.3187,"reference_plane":"Laplace"},{"planet":"Mars","moon":"Deimos","id":"402","a_km":23457.0,"eccentricity":0.0,"inclination_deg":1.8,"period_days":1.2625,"reference_plane":"Laplace"},{"planet":"Jupiter","moon":"Io","id":"501","a_km":421800.0,"eccentricity":0.004,"inclination_deg":0.0,"period_days":1.762732,"reference_plane":"Laplace"},{"planet":"Jupiter","moon":"Europa","id":"502","a_km":671100.0,"eccentricity":0.009,"inclination_deg":0.5,"period_days":3.525463,"reference_plane":"Laplace"},{"planet":"Jupiter","moon":"Ganymede","id":"503","a_km":1070400.0,"eccentricity":0.001,"inclination_deg":0.2,"period_days":7.155588,"reference_plane":"Laplace"},{"planet":"Jupiter","moon":"Callisto","id":"504","a_km":1882700.0,"eccentricity":0.007,"inclination_deg":0.3,"period_days":16.69044,"reference_plane":"Laplace"},{"planet":"Jupiter","moon":"Amalthea","id":"505","a_km":181400.0,"eccentricity":0.003,"inclination_deg":0.4,"period_days":0.499918,"reference_plane":"Laplace"},{"planet":"Jupiter","moon":"Thebe","id":"514","a_km":221900.0,"eccentricity":0.018,"inclination_deg":1.1,"period_days":0.676105,"reference_plane":"Laplace"},{"planet":"Jupiter","moon":"Adrastea","id":"515","a_km":129000.0,"eccentricity":0.0,"inclination_deg":0.0,"period_days":0.29826,"reference_plane":"Laplace"},{"planet":"Jupiter","moon":"Metis","id":"516","a_km":128000.0,"eccentricity":0.0,"inclination_deg":0.0,"period_days":0.294779,"reference_plane":"Laplace"},{"planet":"Jupiter","moon":"Himalia","id":"506","a_km":11439000.0,"eccentricity":0.16,"inclination_deg":28.4,"period_days":249.909,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Elara","id":"507","a_km":11710700.0,"eccentricity":0.212,"inclination_deg":27.8,"period_days":258.8861,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Pasiphae","id":"508","a_km":23463200.0,"eccentricity":0.412,"inclination_deg":148.3,"period_days":734.4215,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Sinope","id":"509","a_km":23679300.0,"eccentricity":0.262,"inclination_deg":157.3,"period_days":744.5951,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Lysithea","id":"510","a_km":11699100.0,"eccentricity":0.117,"inclination_deg":27.7,"period_days":258.5035,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Carme","id":"511","a_km":23139200.0,"eccentricity":0.261,"inclination_deg":164.6,"period_days":719.2806,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Ananke","id":"512","a_km":21029500.0,"eccentricity":0.238,"inclination_deg":147.6,"period_days":623.1097,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Leda","id":"513","a_km":11145200.0,"eccentricity":0.162,"inclination_deg":28.2,"period_days":240.3264,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Callirrhoe","id":"517","a_km":23789400.0,"eccentricity":0.29,"inclination_deg":144.9,"period_days":749.791,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Themisto","id":"518","a_km":7397000.0,"eccentricity":0.257,"inclination_deg":44.3,"period_days":129.9681,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Magaclite","id":"519","a_km":23640100.0,"eccentricity":0.421,"inclination_deg":149.9,"period_days":742.7715,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Taygete","id":"520","a_km":23103400.0,"eccentricity":0.257,"inclination_deg":164.7,"period_days":717.5917,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Chaldene","id":"521","a_km":22926300.0,"eccentricity":0.261,"inclination_deg":164.7,"period_days":709.3625,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Harpalyke","id":"522","a_km":20887500.0,"eccentricity":0.239,"inclination_deg":147.8,"period_days":616.7833,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Kalyke","id":"523","a_km":23298000.0,"eccentricity":0.261,"inclination_deg":164.7,"period_days":726.7007,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Iocaste","id":"524","a_km":21062300.0,"eccentricity":0.223,"inclination_deg":148.7,"period_days":624.5479,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Erinome","id":"525","a_km":23027200.0,"eccentricity":0.272,"inclination_deg":164.3,"period_days":714.0542,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Isonoe","id":"526","a_km":22976300.0,"eccentricity":0.249,"inclination_deg":164.9,"period_days":711.6604,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Praxidike","id":"527","a_km":20931100.0,"eccentricity":0.245,"inclination_deg":148.2,"period_days":618.7229,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Autonoe","id":"528","a_km":23785200.0,"eccentricity":0.326,"inclination_deg":150.7,"period_days":749.6097,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Thyone","id":"529","a_km":20972700.0,"eccentricity":0.235,"inclination_deg":147.6,"period_days":620.5875,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Hermippe","id":"530","a_km":21103600.0,"eccentricity":0.22,"inclination_deg":150.2,"period_days":626.3799,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Aitne","id":"531","a_km":23059400.0,"eccentricity":0.273,"inclination_deg":164.5,"period_days":715.5396,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Eurydome","id":"532","a_km":22894500.0,"eccentricity":0.287,"inclination_deg":148.9,"period_days":707.8569,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Euanthe","id":"533","a_km":20822900.0,"eccentricity":0.243,"inclination_deg":148.1,"period_days":613.9278,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Euporie","id":"534","a_km":19261900.0,"eccentricity":0.148,"inclination_deg":145.5,"period_days":546.1778,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Orthosie","id":"535","a_km":20897800.0,"eccentricity":0.294,"inclination_deg":144.2,"period_days":617.2347,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Sponde","id":"536","a_km":23538700.0,"eccentricity":0.323,"inclination_deg":149.4,"period_days":737.9542,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Kale","id":"537","a_km":23047800.0,"eccentricity":0.262,"inclination_deg":164.6,"period_days":715.016,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Pasithee","id":"538","a_km":22840800.0,"eccentricity":0.274,"inclination_deg":164.5,"period_days":705.409,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Hegemone","id":"539","a_km":23342600.0,"eccentricity":0.357,"inclination_deg":152.5,"period_days":728.7743,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Mneme","id":"540","a_km":20815800.0,"eccentricity":0.24,"inclination_deg":147.8,"period_days":613.6104,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Aoede","id":"541","a_km":23773100.0,"eccentricity":0.437,"inclination_deg":155.7,"period_days":749.0708,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Thelxinoe","id":"542","a_km":20972300.0,"eccentricity":0.229,"inclination_deg":150.7,"period_days":620.5458,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Arche","id":"543","a_km":23093200.0,"eccentricity":0.263,"inclination_deg":164.5,"period_days":717.1056,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Kallichore","id":"544","a_km":23017100.0,"eccentricity":0.253,"inclination_deg":164.7,"period_days":713.5931,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Helike","id":"545","a_km":20911400.0,"eccentricity":0.155,"inclination_deg":154.4,"period_days":617.8625,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Carpo","id":"546","a_km":17039500.0,"eccentricity":0.415,"inclination_deg":53.3,"period_days":454.4,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Eukelade","id":"547","a_km":23062400.0,"eccentricity":0.274,"inclination_deg":164.7,"period_days":715.6868,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Cyllene","id":"548","a_km":23650000.0,"eccentricity":0.421,"inclination_deg":146.8,"period_days":743.2062,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Kore","id":"549","a_km":24203300.0,"eccentricity":0.338,"inclination_deg":141.7,"period_days":769.4229,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Herse","id":"550","a_km":23146700.0,"eccentricity":0.258,"inclination_deg":164.4,"period_days":719.6264,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2010_J_1","id":"551","a_km":23185600.0,"eccentricity":0.256,"inclination_deg":164.5,"period_days":721.4257,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2010_J_2","id":"552","a_km":20786900.0,"eccentricity":0.244,"inclination_deg":148.0,"period_days":612.3507,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Dia","id":"553","a_km":12257900.0,"eccentricity":0.232,"inclination_deg":29.1,"period_days":277.2472,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2016_J_1","id":"554","a_km":20796700.0,"eccentricity":0.245,"inclination_deg":145.1,"period_days":612.7812,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_18","id":"555","a_km":20332800.0,"eccentricity":0.102,"inclination_deg":145.7,"period_days":592.3333,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2011_J_2","id":"556","a_km":22903400.0,"eccentricity":0.358,"inclination_deg":151.7,"period_days":708.2931,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Eirene","id":"557","a_km":23051300.0,"eccentricity":0.263,"inclination_deg":164.7,"period_days":715.191,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Philophrosyn","id":"558","a_km":22600200.0,"eccentricity":0.221,"inclination_deg":146.1,"period_days":694.2042,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_1","id":"559","a_km":23739600.0,"eccentricity":0.321,"inclination_deg":145.6,"period_days":747.4382,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Eupheme","id":"560","a_km":20763400.0,"eccentricity":0.234,"inclination_deg":147.9,"period_days":611.316,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_19","id":"561","a_km":23153100.0,"eccentricity":0.264,"inclination_deg":164.6,"period_days":719.9222,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Valetudo","id":"562","a_km":18690100.0,"eccentricity":0.217,"inclination_deg":34.5,"period_days":522.0743,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_2","id":"563","a_km":22949600.0,"eccentricity":0.27,"inclination_deg":164.5,"period_days":710.4208,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_3","id":"564","a_km":20936500.0,"eccentricity":0.238,"inclination_deg":147.9,"period_days":618.9653,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Pandia","id":"565","a_km":11479600.0,"eccentricity":0.178,"inclination_deg":28.9,"period_days":251.2319,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_5","id":"566","a_km":23202000.0,"eccentricity":0.261,"inclination_deg":164.7,"period_days":722.1972,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_6","id":"567","a_km":23251200.0,"eccentricity":0.333,"inclination_deg":149.6,"period_days":724.4688,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_7","id":"568","a_km":20960400.0,"eccentricity":0.235,"inclination_deg":147.4,"period_days":620.0167,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_8","id":"569","a_km":22819600.0,"eccentricity":0.259,"inclination_deg":164.8,"period_days":704.4181,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_9","id":"570","a_km":21764200.0,"eccentricity":0.197,"inclination_deg":155.4,"period_days":656.0479,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"Ersa","id":"571","a_km":11399400.0,"eccentricity":0.117,"inclination_deg":29.0,"period_days":248.6153,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2011_J_1","id":"572","a_km":23120800.0,"eccentricity":0.269,"inclination_deg":164.7,"period_days":718.4153,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_2","id":"55501","a_km":20992900.0,"eccentricity":0.225,"inclination_deg":150.1,"period_days":621.4715,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_4","id":"55502","a_km":22922300.0,"eccentricity":0.327,"inclination_deg":148.3,"period_days":709.1229,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_9","id":"55503","a_km":23195100.0,"eccentricity":0.268,"inclination_deg":164.7,"period_days":721.8792,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_10","id":"55504","a_km":23384400.0,"eccentricity":0.257,"inclination_deg":164.6,"period_days":730.7375,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_12","id":"55505","a_km":20959300.0,"eccentricity":0.235,"inclination_deg":150.0,"period_days":619.9611,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_16","id":"55506","a_km":20877500.0,"eccentricity":0.238,"inclination_deg":147.8,"period_days":616.3444,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_23","id":"55507","a_km":23824000.0,"eccentricity":0.306,"inclination_deg":144.4,"period_days":751.3993,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2003_J_24","id":"55508","a_km":22882400.0,"eccentricity":0.263,"inclination_deg":164.6,"period_days":707.3347,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2011_J_3","id":"55509","a_km":11716800.0,"eccentricity":0.192,"inclination_deg":27.6,"period_days":259.0875,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2018_J_2","id":"55510","a_km":11419700.0,"eccentricity":0.152,"inclination_deg":28.3,"period_days":249.275,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2018_J_3","id":"55511","a_km":23400200.0,"eccentricity":0.268,"inclination_deg":164.9,"period_days":731.4875,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2021_J_1","id":"55512","a_km":20954700.0,"eccentricity":0.228,"inclination_deg":150.5,"period_days":619.7687,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2021_J_2","id":"55513","a_km":20926600.0,"eccentricity":0.242,"inclination_deg":148.1,"period_days":618.5028,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2021_J_3","id":"55514","a_km":20776600.0,"eccentricity":0.239,"inclination_deg":147.9,"period_days":611.8736,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2021_J_4","id":"55515","a_km":23019700.0,"eccentricity":0.265,"inclination_deg":164.6,"period_days":713.7056,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2021_J_5","id":"55516","a_km":23414600.0,"eccentricity":0.272,"inclination_deg":164.9,"period_days":732.1528,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2021_J_6","id":"55517","a_km":22870400.0,"eccentricity":0.271,"inclination_deg":164.9,"period_days":706.7653,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2016_J_3","id":"55518","a_km":22719300.0,"eccentricity":0.251,"inclination_deg":164.6,"period_days":699.7583,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2016_J_4","id":"55519","a_km":23113900.0,"eccentricity":0.294,"inclination_deg":147.1,"period_days":718.0382,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2018_J_4","id":"55520","a_km":16328500.0,"eccentricity":0.177,"inclination_deg":50.2,"period_days":426.2646,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2022_J_1","id":"55521","a_km":22744700.0,"eccentricity":0.257,"inclination_deg":164.5,"period_days":700.9333,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2022_J_2","id":"55522","a_km":23073400.0,"eccentricity":0.263,"inclination_deg":164.7,"period_days":716.2104,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2022_J_3","id":"55523","a_km":21015100.0,"eccentricity":0.248,"inclination_deg":148.1,"period_days":622.4361,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_10","id":"55525","a_km":21075800.0,"eccentricity":0.209,"inclination_deg":145.1,"period_days":625.1493,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_11","id":"55526","a_km":22991300.0,"eccentricity":0.268,"inclination_deg":164.8,"period_days":712.3826,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2011_J_4","id":"55527","a_km":11104600.0,"eccentricity":0.128,"inclination_deg":28.5,"period_days":239.0521,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2018_J_5","id":"55528","a_km":23269900.0,"eccentricity":0.261,"inclination_deg":164.9,"period_days":725.3785,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2024_J_1","id":"55529","a_km":23462100.0,"eccentricity":0.273,"inclination_deg":164.7,"period_days":734.3764,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2011_J_5","id":"55530","a_km":23527800.0,"eccentricity":0.251,"inclination_deg":164.6,"period_days":737.4646,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2010_J_3","id":"55531","a_km":23862900.0,"eccentricity":0.313,"inclination_deg":148.3,"period_days":753.2771,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2010_J_4","id":"55532","a_km":22793400.0,"eccentricity":0.278,"inclination_deg":164.6,"period_days":703.1938,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2010_J_5","id":"55533","a_km":23581000.0,"eccentricity":0.257,"inclination_deg":164.6,"period_days":739.9875,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2010_J_6","id":"55534","a_km":21489800.0,"eccentricity":0.297,"inclination_deg":149.9,"period_days":643.6715,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2011_J_6","id":"55535","a_km":23238700.0,"eccentricity":0.261,"inclination_deg":164.9,"period_days":723.934,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_12","id":"55536","a_km":23270500.0,"eccentricity":0.257,"inclination_deg":164.8,"period_days":725.3986,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_13","id":"55537","a_km":22842700.0,"eccentricity":0.277,"inclination_deg":164.5,"period_days":705.4972,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_14","id":"55538","a_km":23412500.0,"eccentricity":0.436,"inclination_deg":142.7,"period_days":732.0424,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_15","id":"55539","a_km":23170300.0,"eccentricity":0.232,"inclination_deg":149.2,"period_days":720.6535,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_16","id":"55540","a_km":23007800.0,"eccentricity":0.268,"inclination_deg":164.7,"period_days":713.1292,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_17","id":"55541","a_km":11776100.0,"eccentricity":0.164,"inclination_deg":29.0,"period_days":261.066,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2017_J_18","id":"55542","a_km":22923800.0,"eccentricity":0.254,"inclination_deg":164.9,"period_days":709.2396,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2021_J_7","id":"55543","a_km":23305900.0,"eccentricity":0.253,"inclination_deg":149.4,"period_days":727.0104,"reference_plane":"ecliptic"},{"planet":"Jupiter","moon":"S2021_J_8","id":"55544","a_km":20978900.0,"eccentricity":0.243,"inclination_deg":147.1,"period_days":620.8486,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Mimas","id":"601","a_km":186000.0,"eccentricity":0.02,"inclination_deg":1.6,"period_days":0.942422,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Enceladus","id":"602","a_km":238400.0,"eccentricity":0.005,"inclination_deg":0.0,"period_days":1.370218,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Tethys","id":"603","a_km":295000.0,"eccentricity":0.001,"inclination_deg":1.1,"period_days":1.887802,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Dione","id":"604","a_km":377700.0,"eccentricity":0.002,"inclination_deg":0.0,"period_days":2.736916,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Rhea","id":"605","a_km":527200.0,"eccentricity":0.001,"inclination_deg":0.3,"period_days":4.517503,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Titan","id":"606","a_km":1221900.0,"eccentricity":0.029,"inclination_deg":0.3,"period_days":15.945448,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Hyperion","id":"607","a_km":1481500.0,"eccentricity":0.105,"inclination_deg":0.6,"period_days":21.276658,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Iapetus","id":"608","a_km":3561700.0,"eccentricity":0.028,"inclination_deg":7.6,"period_days":79.331002,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Phoebe","id":"609","a_km":12929400.0,"eccentricity":0.164,"inclination_deg":175.2,"period_days":550.30391,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Helene","id":"612","a_km":377600.0,"eccentricity":0.007,"inclination_deg":0.2,"period_days":2.736916,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Telesto","id":"613","a_km":295000.0,"eccentricity":0.001,"inclination_deg":1.2,"period_days":1.887802,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Calypso","id":"614","a_km":295000.0,"eccentricity":0.001,"inclination_deg":1.5,"period_days":1.887803,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Methone","id":"632","a_km":194700.0,"eccentricity":0.002,"inclination_deg":0.0,"period_days":1.009549,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Polydeuces","id":"634","a_km":377600.0,"eccentricity":0.019,"inclination_deg":0.2,"period_days":2.736916,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Janus","id":"610","a_km":151500.0,"eccentricity":0.007,"inclination_deg":0.2,"period_days":0.697353,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Epimetheus","id":"611","a_km":151400.0,"eccentricity":0.02,"inclination_deg":0.3,"period_days":0.697012,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Atlas","id":"615","a_km":137700.0,"eccentricity":0.001,"inclination_deg":0.0,"period_days":0.604602,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Prometheus","id":"616","a_km":139400.0,"eccentricity":0.002,"inclination_deg":0.0,"period_days":0.615878,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Pandora","id":"617","a_km":141700.0,"eccentricity":0.004,"inclination_deg":0.0,"period_days":0.631369,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Pan","id":"618","a_km":133600.0,"eccentricity":0.0,"inclination_deg":0.0,"period_days":0.575051,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Pallene","id":"633","a_km":212300.0,"eccentricity":0.004,"inclination_deg":0.2,"period_days":1.156059,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Daphnis","id":"635","a_km":136500.0,"eccentricity":0.0,"inclination_deg":0.0,"period_days":0.59408,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Anthe","id":"649","a_km":198100.0,"eccentricity":0.002,"inclination_deg":0.0,"period_days":1.038898,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Aegaeon","id":"653","a_km":167500.0,"eccentricity":0.0,"inclination_deg":0.0,"period_days":0.808115,"reference_plane":"Laplace"},{"planet":"Saturn","moon":"Ymir","id":"619","a_km":22955600.0,"eccentricity":0.338,"inclination_deg":172.3,"period_days":1298.6819,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Paaliaq","id":"620","a_km":14997700.0,"eccentricity":0.378,"inclination_deg":48.5,"period_days":685.7153,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Tarvos","id":"621","a_km":18216600.0,"eccentricity":0.522,"inclination_deg":37.8,"period_days":917.984,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Ijiraq","id":"622","a_km":11344700.0,"eccentricity":0.293,"inclination_deg":49.2,"period_days":451.1201,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Suttungr","id":"623","a_km":19391900.0,"eccentricity":0.116,"inclination_deg":175.7,"period_days":1008.2451,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Kiviuq","id":"624","a_km":11307400.0,"eccentricity":0.275,"inclination_deg":48.0,"period_days":448.9076,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Mundilfari","id":"625","a_km":18588200.0,"eccentricity":0.211,"inclination_deg":167.1,"period_days":946.2903,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Albiorix","id":"626","a_km":16329200.0,"eccentricity":0.482,"inclination_deg":36.8,"period_days":779.0701,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Skathi","id":"627","a_km":15575400.0,"eccentricity":0.281,"inclination_deg":151.6,"period_days":725.7313,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Erriapus","id":"628","a_km":17506900.0,"eccentricity":0.475,"inclination_deg":37.1,"period_days":864.9236,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Siarnaq","id":"629","a_km":17881100.0,"eccentricity":0.308,"inclination_deg":47.8,"period_days":892.6799,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Thrymr","id":"630","a_km":20330500.0,"eccentricity":0.467,"inclination_deg":175.0,"period_days":1082.2292,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Narvi","id":"631","a_km":19285000.0,"eccentricity":0.441,"inclination_deg":142.2,"period_days":999.9417,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Aegir","id":"636","a_km":20664400.0,"eccentricity":0.255,"inclination_deg":166.1,"period_days":1109.1278,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Bebhionn","id":"637","a_km":17027300.0,"eccentricity":0.459,"inclination_deg":38.6,"period_days":829.6437,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Bergelmir","id":"638","a_km":19268100.0,"eccentricity":0.145,"inclination_deg":158.8,"period_days":998.6187,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Bestla","id":"639","a_km":20337800.0,"eccentricity":0.486,"inclination_deg":138.3,"period_days":1082.9444,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Farbauti","id":"640","a_km":20290700.0,"eccentricity":0.249,"inclination_deg":156.2,"period_days":1079.1243,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Fenrir","id":"641","a_km":22330800.0,"eccentricity":0.137,"inclination_deg":164.5,"period_days":1245.9215,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Fornjot","id":"642","a_km":24936800.0,"eccentricity":0.213,"inclination_deg":170.0,"period_days":1470.3618,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Hati","id":"643","a_km":19695000.0,"eccentricity":0.372,"inclination_deg":165.4,"period_days":1032.0236,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Hyrrokkin","id":"644","a_km":18340900.0,"eccentricity":0.336,"inclination_deg":149.9,"period_days":927.4569,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Kari","id":"645","a_km":22032100.0,"eccentricity":0.469,"inclination_deg":153.0,"period_days":1220.9757,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Loge","id":"646","a_km":22919200.0,"eccentricity":0.191,"inclination_deg":168.1,"period_days":1295.5243,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Skoll","id":"647","a_km":17623400.0,"eccentricity":0.463,"inclination_deg":159.4,"period_days":873.5736,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Surtur","id":"648","a_km":22748000.0,"eccentricity":0.448,"inclination_deg":168.4,"period_days":1281.1396,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Jarnsaxa","id":"650","a_km":19273500.0,"eccentricity":0.218,"inclination_deg":163.0,"period_days":999.1271,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Greip","id":"651","a_km":18380000.0,"eccentricity":0.317,"inclination_deg":174.2,"period_days":930.4389,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Tarqeq","id":"652","a_km":17751000.0,"eccentricity":0.144,"inclination_deg":48.7,"period_days":882.8514,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Gridr","id":"654","a_km":19250600.0,"eccentricity":0.187,"inclination_deg":163.9,"period_days":997.3319,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Angrboda","id":"655","a_km":20591500.0,"eccentricity":0.216,"inclination_deg":177.7,"period_days":1103.1986,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Skrymir","id":"656","a_km":21447400.0,"eccentricity":0.437,"inclination_deg":175.6,"period_days":1172.7215,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Gerd","id":"657","a_km":20947500.0,"eccentricity":0.517,"inclination_deg":174.4,"period_days":1131.9062,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_26","id":"658","a_km":26097500.0,"eccentricity":0.147,"inclination_deg":172.9,"period_days":1574.2535,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Eggther","id":"659","a_km":19844600.0,"eccentricity":0.157,"inclination_deg":165.0,"period_days":1043.8035,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_29","id":"660","a_km":17063900.0,"eccentricity":0.485,"inclination_deg":38.6,"period_days":832.2736,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Beli","id":"661","a_km":20703700.0,"eccentricity":0.087,"inclination_deg":158.9,"period_days":1112.2687,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Gunnlod","id":"662","a_km":21141800.0,"eccentricity":0.251,"inclination_deg":160.4,"period_days":1147.7354,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Thiazzi","id":"663","a_km":23577500.0,"eccentricity":0.511,"inclination_deg":158.8,"period_days":1351.8319,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_34","id":"664","a_km":24145800.0,"eccentricity":0.279,"inclination_deg":168.3,"period_days":1400.9299,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Alvaldi","id":"665","a_km":21993800.0,"eccentricity":0.238,"inclination_deg":177.4,"period_days":1217.8042,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"Geirrod","id":"666","a_km":22259400.0,"eccentricity":0.539,"inclination_deg":154.4,"period_days":1240.0465,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_31","id":"65067","a_km":17497100.0,"eccentricity":0.159,"inclination_deg":48.0,"period_days":863.9229,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_24","id":"65070","a_km":23338200.0,"eccentricity":0.071,"inclination_deg":37.4,"period_days":1331.3222,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_28","id":"65077","a_km":21865900.0,"eccentricity":0.159,"inclination_deg":167.9,"period_days":1207.1833,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_21","id":"65079","a_km":23160900.0,"eccentricity":0.394,"inclination_deg":153.2,"period_days":1316.1167,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_36","id":"65081","a_km":23390800.0,"eccentricity":0.625,"inclination_deg":153.3,"period_days":1335.8042,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_37","id":"65082","a_km":15956300.0,"eccentricity":0.448,"inclination_deg":158.2,"period_days":752.5465,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_39","id":"65084","a_km":23192400.0,"eccentricity":0.1,"inclination_deg":165.9,"period_days":1318.7417,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_7","id":"65085","a_km":21327600.0,"eccentricity":0.511,"inclination_deg":164.8,"period_days":1162.9264,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_12","id":"65086","a_km":19801000.0,"eccentricity":0.337,"inclination_deg":164.7,"period_days":1040.3896,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_13","id":"65087","a_km":18453700.0,"eccentricity":0.265,"inclination_deg":169.0,"period_days":936.0889,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_17","id":"65088","a_km":19699000.0,"eccentricity":0.162,"inclination_deg":167.9,"period_days":1032.3694,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_1","id":"65089","a_km":18746300.0,"eccentricity":0.105,"inclination_deg":156.1,"period_days":958.3236,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_3","id":"65090","a_km":21353100.0,"eccentricity":0.432,"inclination_deg":156.1,"period_days":1165.0194,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2007_S_2","id":"65091","a_km":15939100.0,"eccentricity":0.232,"inclination_deg":174.0,"period_days":751.3306,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2007_S_3","id":"65092","a_km":19614400.0,"eccentricity":0.15,"inclination_deg":173.8,"period_days":1025.6986,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_1","id":"65093","a_km":11245400.0,"eccentricity":0.383,"inclination_deg":49.5,"period_days":445.1701,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_2","id":"65094","a_km":16560300.0,"eccentricity":0.279,"inclination_deg":173.3,"period_days":795.6722,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_3","id":"65095","a_km":17077400.0,"eccentricity":0.248,"inclination_deg":166.9,"period_days":833.1917,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_1","id":"65096","a_km":11338600.0,"eccentricity":0.337,"inclination_deg":48.2,"period_days":450.7722,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_2","id":"65097","a_km":17869000.0,"eccentricity":0.152,"inclination_deg":170.7,"period_days":891.8576,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_40","id":"65098","a_km":16075600.0,"eccentricity":0.297,"inclination_deg":169.2,"period_days":760.9993,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_9","id":"65100","a_km":14406700.0,"eccentricity":0.249,"inclination_deg":173.0,"period_days":645.5771,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2007_S_5","id":"65101","a_km":15835600.0,"eccentricity":0.104,"inclination_deg":158.4,"period_days":744.0056,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_3","id":"65102","a_km":18056800.0,"eccentricity":0.142,"inclination_deg":46.0,"period_days":905.841,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_4","id":"65103","a_km":17951900.0,"eccentricity":0.408,"inclination_deg":170.1,"period_days":898.0889,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_41","id":"65104","a_km":18095400.0,"eccentricity":0.301,"inclination_deg":165.7,"period_days":908.8924,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_4","id":"65105","a_km":18236400.0,"eccentricity":0.496,"inclination_deg":40.1,"period_days":919.5174,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_5","id":"65106","a_km":18391000.0,"eccentricity":0.22,"inclination_deg":48.2,"period_days":931.1889,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2007_S_6","id":"65107","a_km":18545000.0,"eccentricity":0.168,"inclination_deg":166.5,"period_days":942.9785,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_42","id":"65108","a_km":18240700.0,"eccentricity":0.158,"inclination_deg":165.7,"period_days":919.8847,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_10","id":"65109","a_km":18979900.0,"eccentricity":0.151,"inclination_deg":161.6,"period_days":976.3438,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_5","id":"65110","a_km":19090100.0,"eccentricity":0.216,"inclination_deg":158.8,"period_days":984.8722,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_43","id":"65111","a_km":18935700.0,"eccentricity":0.432,"inclination_deg":171.1,"period_days":972.8528,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_44","id":"65112","a_km":19515400.0,"eccentricity":0.129,"inclination_deg":167.7,"period_days":1017.9146,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_45","id":"65113","a_km":19693700.0,"eccentricity":0.551,"inclination_deg":154.0,"period_days":1031.8604,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_11","id":"65114","a_km":19711900.0,"eccentricity":0.143,"inclination_deg":174.1,"period_days":1033.3521,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_12","id":"65115","a_km":19570300.0,"eccentricity":0.542,"inclination_deg":38.6,"period_days":1022.2868,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_6","id":"65116","a_km":18205600.0,"eccentricity":0.12,"inclination_deg":46.4,"period_days":917.1056,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_13","id":"65117","a_km":19953300.0,"eccentricity":0.313,"inclination_deg":162.0,"period_days":1052.316,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_7","id":"65118","a_km":20185100.0,"eccentricity":0.233,"inclination_deg":174.2,"period_days":1070.8007,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_8","id":"65119","a_km":20287400.0,"eccentricity":0.311,"inclination_deg":172.8,"period_days":1078.8604,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_9","id":"65120","a_km":20359700.0,"eccentricity":0.433,"inclination_deg":159.5,"period_days":1084.6194,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_46","id":"65121","a_km":20513800.0,"eccentricity":0.249,"inclination_deg":177.2,"period_days":1096.9882,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_10","id":"65122","a_km":20700300.0,"eccentricity":0.248,"inclination_deg":163.9,"period_days":1111.9924,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_47","id":"65123","a_km":16050700.0,"eccentricity":0.291,"inclination_deg":160.9,"period_days":759.2201,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_11","id":"65124","a_km":20664200.0,"eccentricity":0.513,"inclination_deg":144.6,"period_days":1109.1097,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_14","id":"65125","a_km":21062300.0,"eccentricity":0.06,"inclination_deg":166.7,"period_days":1141.2708,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_12","id":"65126","a_km":20895000.0,"eccentricity":0.476,"inclination_deg":167.1,"period_days":1127.5993,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_6","id":"65127","a_km":21253300.0,"eccentricity":0.48,"inclination_deg":166.9,"period_days":1156.8111,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_13","id":"65128","a_km":20964500.0,"eccentricity":0.318,"inclination_deg":177.3,"period_days":1133.2653,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2005_S_4","id":"65129","a_km":11324500.0,"eccentricity":0.315,"inclination_deg":48.0,"period_days":449.925,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2007_S_7","id":"65130","a_km":15931600.0,"eccentricity":0.217,"inclination_deg":169.3,"period_days":750.7958,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2007_S_8","id":"65131","a_km":17048900.0,"eccentricity":0.49,"inclination_deg":36.2,"period_days":831.2111,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_7","id":"65132","a_km":17394000.0,"eccentricity":0.5,"inclination_deg":161.4,"period_days":856.534,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_14","id":"65133","a_km":17852800.0,"eccentricity":0.172,"inclination_deg":46.2,"period_days":890.5868,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_15","id":"65134","a_km":21191100.0,"eccentricity":0.257,"inclination_deg":157.8,"period_days":1151.6611,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2005_S_5","id":"65135","a_km":21364900.0,"eccentricity":0.588,"inclination_deg":169.5,"period_days":1165.9562,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_15","id":"65136","a_km":21799600.0,"eccentricity":0.117,"inclination_deg":161.1,"period_days":1201.6924,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_16","id":"65137","a_km":21721200.0,"eccentricity":0.204,"inclination_deg":164.1,"period_days":1195.1285,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_17","id":"65138","a_km":22384200.0,"eccentricity":0.425,"inclination_deg":168.7,"period_days":1250.4639,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_48","id":"65139","a_km":22137400.0,"eccentricity":0.374,"inclination_deg":161.9,"period_days":1229.8632,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_8","id":"65140","a_km":21967200.0,"eccentricity":0.252,"inclination_deg":161.8,"period_days":1215.6063,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_49","id":"65141","a_km":22399400.0,"eccentricity":0.453,"inclination_deg":159.8,"period_days":1251.6847,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_50","id":"65142","a_km":22345000.0,"eccentricity":0.45,"inclination_deg":164.0,"period_days":1247.1917,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_18","id":"65143","a_km":22760600.0,"eccentricity":0.131,"inclination_deg":169.5,"period_days":1282.0917,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_16","id":"65144","a_km":23265200.0,"eccentricity":0.25,"inclination_deg":162.0,"period_days":1324.9458,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_17","id":"65145","a_km":22722700.0,"eccentricity":0.546,"inclination_deg":155.5,"period_days":1278.9618,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_18","id":"65146","a_km":23139500.0,"eccentricity":0.509,"inclination_deg":154.6,"period_days":1314.2674,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_19","id":"65147","a_km":23044400.0,"eccentricity":0.458,"inclination_deg":151.8,"period_days":1306.1604,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_20","id":"65148","a_km":23677900.0,"eccentricity":0.354,"inclination_deg":156.0,"period_days":1360.4674,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_19","id":"65149","a_km":23800500.0,"eccentricity":0.467,"inclination_deg":175.5,"period_days":1371.0201,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_51","id":"65150","a_km":25207100.0,"eccentricity":0.201,"inclination_deg":171.2,"period_days":1494.4076,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_9","id":"65151","a_km":25408700.0,"eccentricity":0.531,"inclination_deg":161.4,"period_days":1512.384,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_52","id":"65152","a_km":26446400.0,"eccentricity":0.291,"inclination_deg":165.4,"period_days":1605.8868,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2007_S_9","id":"65153","a_km":20174600.0,"eccentricity":0.36,"inclination_deg":159.3,"period_days":1069.8069,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_53","id":"65154","a_km":23279800.0,"eccentricity":0.24,"inclination_deg":162.6,"period_days":1326.184,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_10","id":"65155","a_km":25315300.0,"eccentricity":0.296,"inclination_deg":165.6,"period_days":1503.9681,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_21","id":"65156","a_km":26439500.0,"eccentricity":0.155,"inclination_deg":171.9,"period_days":1605.2799,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_20","id":"65157","a_km":13193700.0,"eccentricity":0.206,"inclination_deg":173.1,"period_days":565.7889,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_54","id":"65158","a_km":11277500.0,"eccentricity":0.373,"inclination_deg":48.1,"period_days":447.141,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_55","id":"65159","a_km":11294700.0,"eccentricity":0.26,"inclination_deg":48.9,"period_days":448.1618,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_56","id":"65160","a_km":13670200.0,"eccentricity":0.339,"inclination_deg":161.6,"period_days":596.6938,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_57","id":"65161","a_km":18150500.0,"eccentricity":0.263,"inclination_deg":167.9,"period_days":913.0736,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_58","id":"65162","a_km":18254500.0,"eccentricity":0.249,"inclination_deg":45.7,"period_days":920.8035,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_59","id":"65163","a_km":19170700.0,"eccentricity":0.262,"inclination_deg":167.3,"period_days":991.1819,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_60","id":"65164","a_km":19517000.0,"eccentricity":0.28,"inclination_deg":173.8,"period_days":1018.1285,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2004_S_61","id":"65165","a_km":20986900.0,"eccentricity":0.466,"inclination_deg":168.4,"period_days":1135.0646,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2005_S_6","id":"65166","a_km":18107300.0,"eccentricity":0.084,"inclination_deg":47.7,"period_days":909.5785,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2005_S_7","id":"65167","a_km":18502500.0,"eccentricity":0.565,"inclination_deg":34.6,"period_days":939.7451,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_21","id":"65168","a_km":14976500.0,"eccentricity":0.204,"inclination_deg":169.8,"period_days":684.2764,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_22","id":"65169","a_km":15109500.0,"eccentricity":0.246,"inclination_deg":172.0,"period_days":693.4146,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_23","id":"65170","a_km":18269700.0,"eccentricity":0.19,"inclination_deg":43.8,"period_days":921.8632,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_24","id":"65171","a_km":18210700.0,"eccentricity":0.352,"inclination_deg":165.9,"period_days":917.559,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_25","id":"65172","a_km":18572400.0,"eccentricity":0.303,"inclination_deg":158.8,"period_days":945.0681,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_26","id":"65173","a_km":18619300.0,"eccentricity":0.248,"inclination_deg":171.9,"period_days":948.6722,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_27","id":"65174","a_km":19205700.0,"eccentricity":0.14,"inclination_deg":170.5,"period_days":993.7889,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_28","id":"65175","a_km":21955100.0,"eccentricity":0.21,"inclination_deg":172.9,"period_days":1214.5257,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2006_S_29","id":"65176","a_km":25212100.0,"eccentricity":0.239,"inclination_deg":156.2,"period_days":1494.7833,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2007_S_10","id":"65177","a_km":11364900.0,"eccentricity":0.367,"inclination_deg":45.8,"period_days":452.3576,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2007_S_11","id":"65178","a_km":17434400.0,"eccentricity":0.499,"inclination_deg":35.5,"period_days":859.5264,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_22","id":"65179","a_km":11305100.0,"eccentricity":0.369,"inclination_deg":47.3,"period_days":448.775,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_23","id":"65180","a_km":11310200.0,"eccentricity":0.255,"inclination_deg":48.7,"period_days":449.0806,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_24","id":"65181","a_km":11360500.0,"eccentricity":0.345,"inclination_deg":46.7,"period_days":452.0743,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_25","id":"65182","a_km":11329400.0,"eccentricity":0.271,"inclination_deg":48.1,"period_days":450.2181,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_26","id":"65183","a_km":11390900.0,"eccentricity":0.365,"inclination_deg":48.1,"period_days":453.8882,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_27","id":"65184","a_km":16267000.0,"eccentricity":0.42,"inclination_deg":162.1,"period_days":774.6333,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_28","id":"65185","a_km":17496000.0,"eccentricity":0.199,"inclination_deg":158.4,"period_days":864.0938,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_29","id":"65186","a_km":17353900.0,"eccentricity":0.441,"inclination_deg":37.7,"period_days":853.6229,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_30","id":"65187","a_km":17709900.0,"eccentricity":0.107,"inclination_deg":168.3,"period_days":879.9708,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_31","id":"65188","a_km":17739100.0,"eccentricity":0.488,"inclination_deg":39.8,"period_days":882.241,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_32","id":"65189","a_km":17960500.0,"eccentricity":0.276,"inclination_deg":46.2,"period_days":898.7083,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_33","id":"65190","a_km":18696100.0,"eccentricity":0.289,"inclination_deg":170.4,"period_days":954.534,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_34","id":"65191","a_km":18446800.0,"eccentricity":0.536,"inclination_deg":37.6,"period_days":935.4472,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_35","id":"65192","a_km":18557800.0,"eccentricity":0.577,"inclination_deg":157.3,"period_days":943.9993,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_36","id":"65193","a_km":19903200.0,"eccentricity":0.161,"inclination_deg":166.9,"period_days":1048.4188,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_37","id":"65194","a_km":19996900.0,"eccentricity":0.404,"inclination_deg":149.9,"period_days":1055.7771,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_38","id":"65195","a_km":21998400.0,"eccentricity":0.399,"inclination_deg":163.0,"period_days":1218.2743,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_39","id":"65196","a_km":23784500.0,"eccentricity":0.098,"inclination_deg":174.5,"period_days":1369.6437,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_40","id":"65197","a_km":24087800.0,"eccentricity":0.088,"inclination_deg":161.8,"period_days":1395.8771,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_41","id":"65198","a_km":24493600.0,"eccentricity":0.257,"inclination_deg":157.1,"period_days":1431.2979,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_42","id":"65199","a_km":24111600.0,"eccentricity":0.121,"inclination_deg":163.2,"period_days":1397.975,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_43","id":"65200","a_km":26664100.0,"eccentricity":0.277,"inclination_deg":165.3,"period_days":1625.9,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2019_S_44","id":"65201","a_km":26796900.0,"eccentricity":0.512,"inclination_deg":172.6,"period_days":1638.1104,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_11","id":"65202","a_km":11295600.0,"eccentricity":0.372,"inclination_deg":48.2,"period_days":448.2111,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_12","id":"65203","a_km":11314500.0,"eccentricity":0.26,"inclination_deg":50.8,"period_days":449.3271,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_13","id":"65204","a_km":11415600.0,"eccentricity":0.373,"inclination_deg":48.0,"period_days":455.3875,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_14","id":"65205","a_km":16186200.0,"eccentricity":0.313,"inclination_deg":161.7,"period_days":768.8625,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_15","id":"65206","a_km":16729200.0,"eccentricity":0.462,"inclination_deg":37.1,"period_days":807.8243,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_16","id":"65207","a_km":16963400.0,"eccentricity":0.405,"inclination_deg":167.3,"period_days":824.9229,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_17","id":"65208","a_km":17094200.0,"eccentricity":0.378,"inclination_deg":148.9,"period_days":834.4472,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_18","id":"65209","a_km":17777900.0,"eccentricity":0.18,"inclination_deg":168.9,"period_days":885.1153,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_19","id":"65210","a_km":17726700.0,"eccentricity":0.159,"inclination_deg":48.1,"period_days":881.0431,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_20","id":"65211","a_km":17997300.0,"eccentricity":0.133,"inclination_deg":169.8,"period_days":901.525,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_21","id":"65212","a_km":18862100.0,"eccentricity":0.307,"inclination_deg":169.9,"period_days":967.2604,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_22","id":"65213","a_km":19443000.0,"eccentricity":0.059,"inclination_deg":161.3,"period_days":1012.241,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_23","id":"65214","a_km":19801500.0,"eccentricity":0.089,"inclination_deg":165.0,"period_days":1040.3757,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_24","id":"65215","a_km":20618300.0,"eccentricity":0.23,"inclination_deg":159.6,"period_days":1105.4083,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_25","id":"65216","a_km":20763700.0,"eccentricity":0.316,"inclination_deg":171.8,"period_days":1117.0889,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_26","id":"65217","a_km":21264400.0,"eccentricity":0.273,"inclination_deg":163.2,"period_days":1157.766,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_27","id":"65218","a_km":21802300.0,"eccentricity":0.255,"inclination_deg":145.3,"period_days":1202.016,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_28","id":"65219","a_km":21993700.0,"eccentricity":0.474,"inclination_deg":160.1,"period_days":1217.8104,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_29","id":"65220","a_km":22301400.0,"eccentricity":0.047,"inclination_deg":169.1,"period_days":1243.4493,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_30","id":"65221","a_km":21790700.0,"eccentricity":0.601,"inclination_deg":154.2,"period_days":1201.0187,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_31","id":"65222","a_km":22457300.0,"eccentricity":0.238,"inclination_deg":163.8,"period_days":1256.5035,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_32","id":"65223","a_km":21884100.0,"eccentricity":0.502,"inclination_deg":169.1,"period_days":1208.8486,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_33","id":"65224","a_km":22922500.0,"eccentricity":0.555,"inclination_deg":162.8,"period_days":1295.85,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_34","id":"65225","a_km":22435600.0,"eccentricity":0.154,"inclination_deg":160.6,"period_days":1254.6069,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_35","id":"65226","a_km":23030300.0,"eccentricity":0.225,"inclination_deg":174.9,"period_days":1304.9653,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_36","id":"65227","a_km":22806200.0,"eccentricity":0.336,"inclination_deg":168.8,"period_days":1286.0292,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_37","id":"65228","a_km":23751800.0,"eccentricity":0.344,"inclination_deg":174.8,"period_days":1366.8215,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_38","id":"65229","a_km":23583900.0,"eccentricity":0.513,"inclination_deg":159.7,"period_days":1352.4347,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_39","id":"65230","a_km":24262400.0,"eccentricity":0.305,"inclination_deg":160.1,"period_days":1411.1451,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_40","id":"65231","a_km":23785900.0,"eccentricity":0.412,"inclination_deg":167.3,"period_days":1369.7562,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_41","id":"65232","a_km":25876400.0,"eccentricity":0.402,"inclination_deg":160.2,"period_days":1554.3993,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_42","id":"65233","a_km":25329400.0,"eccentricity":0.506,"inclination_deg":157.5,"period_days":1505.3118,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_43","id":"65234","a_km":26657400.0,"eccentricity":0.203,"inclination_deg":164.6,"period_days":1625.2931,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_44","id":"65235","a_km":27259400.0,"eccentricity":0.199,"inclination_deg":168.5,"period_days":1680.641,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_1","id":"65236","a_km":11205400.0,"eccentricity":0.386,"inclination_deg":48.8,"period_days":442.8632,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_2","id":"65237","a_km":11309900.0,"eccentricity":0.339,"inclination_deg":45.7,"period_days":449.0535,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_3","id":"65238","a_km":17646400.0,"eccentricity":0.178,"inclination_deg":46.9,"period_days":875.0007,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_4","id":"65239","a_km":17764600.0,"eccentricity":0.276,"inclination_deg":170.0,"period_days":884.1111,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_5","id":"65240","a_km":25583500.0,"eccentricity":0.599,"inclination_deg":168.8,"period_days":1528.0368,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_6","id":"65241","a_km":11953100.0,"eccentricity":0.336,"inclination_deg":47.4,"period_days":487.909,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_7","id":"65242","a_km":12133700.0,"eccentricity":0.284,"inclination_deg":44.7,"period_days":499.0097,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_8","id":"65243","a_km":14018800.0,"eccentricity":0.122,"inclination_deg":166.9,"period_days":619.691,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_9","id":"65244","a_km":13167500.0,"eccentricity":0.141,"inclination_deg":172.2,"period_days":564.1139,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_10","id":"65245","a_km":15500200.0,"eccentricity":0.302,"inclination_deg":163.0,"period_days":720.4931,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_11","id":"65246","a_km":14046100.0,"eccentricity":0.3,"inclination_deg":170.9,"period_days":621.4931,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_12","id":"65247","a_km":15805900.0,"eccentricity":0.601,"inclination_deg":168.8,"period_days":741.9167,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_13","id":"65248","a_km":15193000.0,"eccentricity":0.179,"inclination_deg":168.5,"period_days":699.1799,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_14","id":"65249","a_km":16853000.0,"eccentricity":0.497,"inclination_deg":171.6,"period_days":816.8618,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_15","id":"65250","a_km":18241300.0,"eccentricity":0.549,"inclination_deg":161.9,"period_days":919.9347,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_16","id":"65251","a_km":17005300.0,"eccentricity":0.27,"inclination_deg":162.6,"period_days":827.9111,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_17","id":"65252","a_km":17385300.0,"eccentricity":0.498,"inclination_deg":35.9,"period_days":855.9354,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_18","id":"65253","a_km":17381700.0,"eccentricity":0.448,"inclination_deg":36.7,"period_days":855.6535,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_19","id":"65254","a_km":17590300.0,"eccentricity":0.092,"inclination_deg":48.2,"period_days":870.9229,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_20","id":"65255","a_km":17261000.0,"eccentricity":0.442,"inclination_deg":136.5,"period_days":846.7653,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_21","id":"65256","a_km":17755400.0,"eccentricity":0.077,"inclination_deg":157.3,"period_days":883.3083,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_22","id":"65257","a_km":18577500.0,"eccentricity":0.182,"inclination_deg":47.5,"period_days":945.3701,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_23","id":"65258","a_km":18783700.0,"eccentricity":0.35,"inclination_deg":164.8,"period_days":961.2236,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_24","id":"65259","a_km":18351800.0,"eccentricity":0.374,"inclination_deg":169.7,"period_days":928.2451,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_25","id":"65260","a_km":19136600.0,"eccentricity":0.281,"inclination_deg":166.4,"period_days":988.5278,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_26","id":"65261","a_km":19894300.0,"eccentricity":0.306,"inclination_deg":163.9,"period_days":1047.7618,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_27","id":"65262","a_km":19820100.0,"eccentricity":0.652,"inclination_deg":151.1,"period_days":1041.8521,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_28","id":"65263","a_km":19881000.0,"eccentricity":0.575,"inclination_deg":168.7,"period_days":1046.5556,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_29","id":"65264","a_km":20042400.0,"eccentricity":0.141,"inclination_deg":172.2,"period_days":1059.4208,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_30","id":"65265","a_km":18238300.0,"eccentricity":0.493,"inclination_deg":142.4,"period_days":919.7139,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_31","id":"65266","a_km":20729200.0,"eccentricity":0.182,"inclination_deg":163.0,"period_days":1114.2889,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_32","id":"65267","a_km":20454400.0,"eccentricity":0.037,"inclination_deg":169.8,"period_days":1092.2417,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_33","id":"65268","a_km":21621900.0,"eccentricity":0.665,"inclination_deg":155.8,"period_days":1187.0667,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_34","id":"65269","a_km":20803900.0,"eccentricity":0.57,"inclination_deg":168.4,"period_days":1120.3833,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_35","id":"65270","a_km":22269700.0,"eccentricity":0.151,"inclination_deg":168.5,"period_days":1240.7618,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_36","id":"65271","a_km":22230600.0,"eccentricity":0.359,"inclination_deg":166.3,"period_days":1237.6104,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_37","id":"65272","a_km":19889800.0,"eccentricity":0.215,"inclination_deg":172.3,"period_days":1047.3903,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_38","id":"65273","a_km":12823500.0,"eccentricity":0.909,"inclination_deg":149.2,"period_days":546.3146,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_39","id":"65274","a_km":20824500.0,"eccentricity":0.124,"inclination_deg":164.8,"period_days":1121.9938,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_40","id":"65275","a_km":21065100.0,"eccentricity":0.342,"inclination_deg":169.6,"period_days":1141.4813,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_41","id":"65276","a_km":21286400.0,"eccentricity":0.279,"inclination_deg":172.1,"period_days":1159.516,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_42","id":"65277","a_km":21837000.0,"eccentricity":0.059,"inclination_deg":166.7,"period_days":1204.8146,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_43","id":"65278","a_km":22563900.0,"eccentricity":0.264,"inclination_deg":170.3,"period_days":1265.5667,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_44","id":"65279","a_km":19292400.0,"eccentricity":0.434,"inclination_deg":167.4,"period_days":1000.4722,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_45","id":"65280","a_km":23438400.0,"eccentricity":0.633,"inclination_deg":157.4,"period_days":1339.8472,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_46","id":"65281","a_km":24708900.0,"eccentricity":0.336,"inclination_deg":143.2,"period_days":1450.2715,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_47","id":"65282","a_km":25102300.0,"eccentricity":0.101,"inclination_deg":162.5,"period_days":1485.0417,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_48","id":"65283","a_km":20029200.0,"eccentricity":0.022,"inclination_deg":169.7,"period_days":1058.3493,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_49","id":"65284","a_km":21766500.0,"eccentricity":0.026,"inclination_deg":171.7,"period_days":1198.9896,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_50","id":"65285","a_km":11656500.0,"eccentricity":0.263,"inclination_deg":166.1,"period_days":469.8201,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_45","id":"65286","a_km":23507700.0,"eccentricity":0.199,"inclination_deg":172.8,"period_days":1345.7917,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_46","id":"65287","a_km":18892100.0,"eccentricity":0.207,"inclination_deg":167.3,"period_days":969.5965,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_47","id":"65288","a_km":21397400.0,"eccentricity":0.564,"inclination_deg":146.1,"period_days":1168.5778,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_48","id":"65289","a_km":11355100.0,"eccentricity":0.373,"inclination_deg":45.9,"period_days":451.7528,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_51","id":"65290","a_km":23431500.0,"eccentricity":0.191,"inclination_deg":163.3,"period_days":1339.2104,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_52","id":"65291","a_km":22528000.0,"eccentricity":0.124,"inclination_deg":146.2,"period_days":1262.3438,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_53","id":"65292","a_km":17181100.0,"eccentricity":0.103,"inclination_deg":171.2,"period_days":840.8181,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_54","id":"65293","a_km":17485100.0,"eccentricity":0.48,"inclination_deg":37.8,"period_days":863.3486,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_55","id":"65294","a_km":16875100.0,"eccentricity":0.491,"inclination_deg":35.9,"period_days":818.5125,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_56","id":"65295","a_km":11287500.0,"eccentricity":0.358,"inclination_deg":45.4,"period_days":447.7458,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_57","id":"65296","a_km":20536100.0,"eccentricity":0.245,"inclination_deg":168.0,"period_days":1098.8285,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2020_S_49","id":"65297","a_km":11305900.0,"eccentricity":0.373,"inclination_deg":48.0,"period_days":448.8382,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_58","id":"65298","a_km":17206000.0,"eccentricity":0.093,"inclination_deg":169.6,"period_days":842.6812,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_59","id":"65299","a_km":20064000.0,"eccentricity":0.467,"inclination_deg":169.5,"period_days":1061.041,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_60","id":"65300","a_km":17493700.0,"eccentricity":0.206,"inclination_deg":170.7,"period_days":863.9632,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_61","id":"65301","a_km":18067700.0,"eccentricity":0.557,"inclination_deg":158.0,"period_days":906.8486,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_62","id":"65302","a_km":14025900.0,"eccentricity":0.467,"inclination_deg":155.6,"period_days":620.1486,"reference_plane":"ecliptic"},{"planet":"Saturn","moon":"S2023_S_63","id":"65303","a_km":18482600.0,"eccentricity":0.266,"inclination_deg":165.0,"period_days":938.2549,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"Ariel","id":"701","a_km":190929.0,"eccentricity":0.001,"inclination_deg":0.0,"period_days":2.520379,"reference_plane":"equatorial"},{"planet":"Uranus","moon":"Umbriel","id":"702","a_km":265986.0,"eccentricity":0.004,"inclination_deg":0.1,"period_days":4.144177,"reference_plane":"equatorial"},{"planet":"Uranus","moon":"Titania","id":"703","a_km":436298.0,"eccentricity":0.002,"inclination_deg":0.1,"period_days":8.705869,"reference_plane":"equatorial"},{"planet":"Uranus","moon":"Oberon","id":"704","a_km":583511.0,"eccentricity":0.002,"inclination_deg":0.1,"period_days":13.463237,"reference_plane":"equatorial"},{"planet":"Uranus","moon":"Miranda","id":"705","a_km":129846.0,"eccentricity":0.001,"inclination_deg":4.4,"period_days":1.413479,"reference_plane":"equatorial"},{"planet":"Uranus","moon":"Puck","id":"715","a_km":86004.0,"eccentricity":0.0,"inclination_deg":0.3,"period_days":0.761833,"reference_plane":"equatorial"},{"planet":"Uranus","moon":"Cordelia","id":"706","a_km":49755.0,"eccentricity":0.0,"inclination_deg":0.2,"period_days":0.3347,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Ophelia","id":"707","a_km":53765.0,"eccentricity":0.011,"inclination_deg":0.2,"period_days":0.3764,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Bianca","id":"708","a_km":59170.0,"eccentricity":0.006,"inclination_deg":2.3,"period_days":0.4347,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Cressida","id":"709","a_km":61770.0,"eccentricity":0.004,"inclination_deg":1.8,"period_days":0.4639,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Desdemona","id":"710","a_km":62663.0,"eccentricity":0.007,"inclination_deg":3.1,"period_days":0.4736,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Juliet","id":"711","a_km":64362.0,"eccentricity":0.006,"inclination_deg":3.0,"period_days":0.4931,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Portia","id":"712","a_km":66101.0,"eccentricity":0.004,"inclination_deg":2.7,"period_days":0.5132,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Rosalind","id":"713","a_km":69930.0,"eccentricity":0.003,"inclination_deg":1.7,"period_days":0.5583,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Belinda","id":"714","a_km":75258.0,"eccentricity":0.002,"inclination_deg":1.4,"period_days":0.6236,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Puck","id":"715","a_km":86007.0,"eccentricity":0.009,"inclination_deg":1.1,"period_days":0.7618,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Perdita","id":"725","a_km":76418.0,"eccentricity":0.005,"inclination_deg":1.6,"period_days":0.6382,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Mab","id":"726","a_km":97737.0,"eccentricity":0.006,"inclination_deg":1.8,"period_days":0.9229,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Cupid","id":"727","a_km":74396.0,"eccentricity":0.007,"inclination_deg":2.0,"period_days":0.6125,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"S2025_U_1","id":"75052","a_km":57844.0,"eccentricity":0.039,"inclination_deg":4.0,"period_days":0.4201,"reference_plane":"Laplace"},{"planet":"Uranus","moon":"Caliban","id":"716","a_km":7167000.0,"eccentricity":0.2,"inclination_deg":141.4,"period_days":580.0,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"Sycorax","id":"717","a_km":12193200.0,"eccentricity":0.52,"inclination_deg":157.0,"period_days":1286.0,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"Prospero","id":"718","a_km":16221000.0,"eccentricity":0.441,"inclination_deg":149.4,"period_days":1974.0,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"Setebos","id":"719","a_km":17519800.0,"eccentricity":0.579,"inclination_deg":153.9,"period_days":2215.0,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"Stephano","id":"720","a_km":7951400.0,"eccentricity":0.235,"inclination_deg":143.6,"period_days":677.0,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"Trinculo","id":"721","a_km":8502600.0,"eccentricity":0.22,"inclination_deg":167.1,"period_days":749.0,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"Francisco","id":"722","a_km":4275700.0,"eccentricity":0.144,"inclination_deg":146.8,"period_days":267.0,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"Margaret","id":"723","a_km":14425000.0,"eccentricity":0.642,"inclination_deg":60.5,"period_days":1655.0,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"Ferdinand","id":"724","a_km":20421400.0,"eccentricity":0.395,"inclination_deg":169.2,"period_days":2788.0,"reference_plane":"ecliptic"},{"planet":"Uranus","moon":"S2023_U1","id":"75051","a_km":7976600.0,"eccentricity":0.25,"inclination_deg":143.9,"period_days":681.0,"reference_plane":"ecliptic"},{"planet":"Neptune","moon":"Triton","id":"801","a_km":354800.0,"eccentricity":0.0,"inclination_deg":157.3,"period_days":5.876994,"reference_plane":"Laplace"},{"planet":"Neptune","moon":"Naiad","id":"803","a_km":48200.0,"eccentricity":0.0,"inclination_deg":4.7,"period_days":0.29398,"reference_plane":"Laplace"},{"planet":"Neptune","moon":"Thalassa","id":"804","a_km":50100.0,"eccentricity":0.0,"inclination_deg":0.2,"period_days":0.311078,"reference_plane":"Laplace"},{"planet":"Neptune","moon":"Despina","id":"805","a_km":52500.0,"eccentricity":0.0,"inclination_deg":0.0,"period_days":0.334656,"reference_plane":"Laplace"},{"planet":"Neptune","moon":"Galatea","id":"806","a_km":62000.0,"eccentricity":0.0,"inclination_deg":0.0,"period_days":0.428744,"reference_plane":"Laplace"},{"planet":"Neptune","moon":"Larissa","id":"807","a_km":73500.0,"eccentricity":0.001,"inclination_deg":0.2,"period_days":0.554989,"reference_plane":"Laplace"},{"planet":"Neptune","moon":"Proteus","id":"808","a_km":117600.0,"eccentricity":0.0,"inclination_deg":0.0,"period_days":1.122315,"reference_plane":"Laplace"},{"planet":"Neptune","moon":"Hippocamp","id":"814","a_km":105300.0,"eccentricity":0.001,"inclination_deg":0.3,"period_days":0.95039,"reference_plane":"Laplace"},{"planet":"Neptune","moon":"Nereid","id":"802","a_km":5513900.0,"eccentricity":0.751,"inclination_deg":5.1,"period_days":360.133039,"reference_plane":"ecliptic"},{"planet":"Neptune","moon":"Halimede","id":"809","a_km":16590500.0,"eccentricity":0.521,"inclination_deg":119.6,"period_days":1879.0,"reference_plane":"ecliptic"},{"planet":"Neptune","moon":"Psamathe","id":"810","a_km":47646600.0,"eccentricity":0.413,"inclination_deg":127.8,"period_days":9149.0,"reference_plane":"ecliptic"},{"planet":"Neptune","moon":"Sao","id":"811","a_km":22239900.0,"eccentricity":0.296,"inclination_deg":50.2,"period_days":2919.0,"reference_plane":"ecliptic"},{"planet":"Neptune","moon":"Laomedeia","id":"812","a_km":23499900.0,"eccentricity":0.419,"inclination_deg":36.9,"period_days":3168.0,"reference_plane":"ecliptic"},{"planet":"Neptune","moon":"Neso","id":"813","a_km":49897800.0,"eccentricity":0.455,"inclination_deg":128.4,"period_days":9805.0,"reference_plane":"ecliptic"},{"planet":"Neptune","moon":"S2002_N5","id":"85051","a_km":23414700.0,"eccentricity":0.433,"inclination_deg":46.3,"period_days":3151.0,"reference_plane":"ecliptic"},{"planet":"Neptune","moon":"S2021_N1","id":"85052","a_km":50700200.0,"eccentricity":0.503,"inclination_deg":135.2,"period_days":10043.0,"reference_plane":"ecliptic"},{"planet":"Pluto","moon":"Charon","id":"901","a_km":19600.0,"eccentricity":0.0,"inclination_deg":0.0,"period_days":6.387222,"reference_plane":"equatorial"},{"planet":"Pluto","moon":"Styx","id":"905","a_km":43200.0,"eccentricity":0.025,"inclination_deg":0.0,"period_days":20.16,"reference_plane":"equatorial"},{"planet":"Pluto","moon":"Nix","id":"902","a_km":49300.0,"eccentricity":0.015,"inclination_deg":0.0,"period_days":24.85,"reference_plane":"equatorial"},{"planet":"Pluto","moon":"Kerberos","id":"904","a_km":58300.0,"eccentricity":0.01,"inclination_deg":0.4,"period_days":32.17,"reference_plane":"equatorial"},{"planet":"Pluto","moon":"Hydra","id":"903","a_km":65200.0,"eccentricity":0.009,"inclination_deg":0.3,"period_days":38.2,"reference_plane":"equatorial"}]'
moons = pd.read_json(io.StringIO(MOON_CATALOGUE_JSON),dtype={'id':str})
MOON_CATALOGUE_SOURCE = 'https://ssd.jpl.nasa.gov/sats/elem/'
MOON_CATALOGUE_DATE = '2026-09-10'
if REFRESH_MOON_CATALOGUE:
    try:
        response=requests.get(MOON_CATALOGUE_SOURCE,timeout=45);response.raise_for_status()
        candidate=pd.read_html(io.StringIO(response.text))[0]
        candidate=candidate[['Planet','Satellite','Code','a (km)','e','i (deg)','P (days)','Frame']]
        candidate.columns=moons.columns
        candidate['id']=candidate['id'].astype(str)
        if len(candidate)<100: raise ValueError('Unexpected catalogue layout; kept embedded snapshot.')
        moons=candidate
        MOON_CATALOGUE_DATE=datetime.now(timezone.utc).isoformat()
    except Exception as exc:
        print('Refresh failed; using embedded snapshot:',str(exc)[:180])
raw_moon_rows=len(moons)
moons=moons.drop_duplicates(subset=['id'],keep='first').reset_index(drop=True)
print(f'Catalogue rows: {raw_moon_rows}; distinct moons: {len(moons)} (duplicate reference-plane rows removed).')
for col in ['a_km','eccentricity','inclination_deg','period_days']:
    moons[col]=pd.to_numeric(moons[col],errors='coerce')
moons['nominal_circular_speed_km_s']=2*np.pi*moons.a_km/(moons.period_days.abs()*DAY_S)
print(f'{len(moons)} JPL table entries; snapshot {MOON_CATALOGUE_DATE}.')
print('Counts are entries in this table, not a claim about the total number discovered.')
display(moons.groupby('planet').size().rename('catalogue_entries').to_frame())
display(moons.head(12))

Catalogue rows: 460; distinct moons: 459 (duplicate reference-plane rows removed).
459 JPL table entries; snapshot 2026-09-10.
Counts are entries in this table, not a claim about the total number discovered.


,catalogue_entries
planet,
Earth,1
Jupiter,115
Mars,2
Neptune,16
Pluto,5
Saturn,291
Uranus,29


,planet,moon,id,a_km,eccentricity,inclination_deg,period_days,reference_plane,nominal_circular_speed_km_s
0,Earth,Moon,301,384400,0.0554,5.16,27.322000,ecliptic,1.023145
1,Mars,Phobos,401,9375,0.0150,1.10,0.318700,Laplace,2.139219
2,Mars,Deimos,402,23457,0.0000,1.80,1.262500,Laplace,1.351161
3,Jupiter,Io,501,421800,0.0040,0.00,1.762732,Laplace,17.401489
4,Jupiter,Europa,502,671100,0.0090,0.50,3.525463,Laplace,13.843223
5,Jupiter,Ganymede,503,1070400,0.0010,0.20,7.155588,Laplace,10.878447
6,Jupiter,Callisto,504,1882700,0.0070,0.30,16.690440,Laplace,8.203128
7,Jupiter,Amalthea,505,181400,0.0030,0.40,0.499918,Laplace,26.387888
8,Jupiter,Thebe,514,221900,0.0180,1.10,0.676105,Laplace,23.867629
9,Jupiter,Adrastea,515,129000,0.0000,0.00,0.298260,Laplace,31.452909


## Cell 10 · Moon orbit sizes around every parent

These seven views cover **every entry with a usable radius and period**, without hiding the irregular outer moons. They use measured mean radii, but all rings are drawn in a common plane for teaching. A circle is not the true eccentric orbit. No date is assigned to the arbitrary marker phases.

In [10]:
MAJOR_IDS = {'Earth':['301'],'Mars':['401','402'],
 'Jupiter':['501','502','503','504'],
 'Saturn':['601','602','603','604','605','606','607','608','609'],
 'Uranus':['701','702','703','704','705'],
 'Neptune':['801','802','808'],'Pluto':['901','902','903','904','905']}
PARENT_IDS={'Mercury':'199','Venus':'299','Earth':'399','Mars':'499','Jupiter':'599',
            'Saturn':'699','Uranus':'799','Neptune':'899','Pluto':'999'}

# All circular rings are intentionally coplanar; no fabricated inclination or phase is implied.
for parent in ['Earth','Mars','Jupiter','Saturn','Uranus','Neptune','Pluto']:
    sub=moons.query('planet == @parent').dropna(subset=['a_km','period_days'])
    theta=np.linspace(0,2*np.pi,100)
    xx=[];yy=[];zz=[]
    for a in sub.a_km/1000:
        xx.extend((a*np.cos(theta)).tolist()+[None]);yy.extend((a*np.sin(theta)).tolist()+[None]);zz.extend([0]*len(theta)+[None])
    fig=go.Figure(go.Scatter3d(x=xx,y=yy,z=zz,mode='lines',line=dict(width=1,color='#667ba0'),hoverinfo='skip',name='Mean-radius circles'))
    phase=np.linspace(0,2*np.pi,len(sub),endpoint=False)
    radius=sub.a_km.to_numpy()/1000
    fig.add_trace(go.Scatter3d(x=radius*np.cos(phase),y=radius*np.sin(phase),z=np.zeros(len(sub)),
        mode='markers',text=sub.moon,customdata=sub[['a_km','period_days','nominal_circular_speed_km_s']].to_numpy(),
        marker=dict(size=3,color=np.log10(sub.a_km),colorscale='Viridis',showscale=False),
        hovertemplate='%{text}<br>Mean a: %{customdata[0]:,.0f} km<br>Period: %{customdata[1]:.3f} d<br>Circular model: %{customdata[2]:.3f} km/s<extra></extra>',name='Catalogue moons'))
    fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',marker=dict(size=9,color='gold'),name=parent))
    style3d(fig,f'{parent}: {len(sub)} catalogue moons · SCHEMATIC radii, coplanar rings, arbitrary phases','1000 km')
    show(fig,'moon_radii_'+parent.lower())
print('These are rotatable size diagrams, not real 3D moon positions. Horizons below gives the actual orbital planes.')

These are rotatable size diagrams, not real 3D moon positions. Horizons below gives the actual orbital planes.


## Cell 11 · Animate a selected moon system

Change `MOON_PLANET` and `MOON_DISPLAY` in Cell 03, then rerun this cell. The numerical periods differ correctly, but the orbit shapes, phases and motion directions are schematic. This does **not** show retrograde moons accurately. The Horizons animation below does.

In [11]:
def moon_demo(parent,selection='major'):
    sub=moons.query('planet == @parent').dropna(subset=['a_km','period_days']).copy()
    if selection=='major':sub=sub[sub.id.isin(MAJOR_IDS.get(parent,[]))]
    if sub.empty:print('No moons selected for',parent);return
    sub=sub.reset_index(drop=True)
    # Use 2 periods of the fastest selected moon to avoid misleading temporal aliasing.
    days=2*sub.period_days.abs().min();times=np.linspace(0,days,121)
    radius=sub.a_km.to_numpy()/1000
    phase0=np.linspace(0,2*np.pi,len(sub),endpoint=False)
    theta=np.linspace(0,2*np.pi,180)
    fig=go.Figure()
    for _,row in sub.iterrows():
        r=row.a_km/1000
        fig.add_trace(line3d(np.c_[r*np.cos(theta),r*np.sin(theta),np.zeros(len(theta))],row.moon+' circular model',width=1,showlegend=False))
    fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',name=parent,marker=dict(size=10,color='gold')))
    # Direction is intentionally illustrative: all prograde; real retrograde orbits need Horizons.
    def points(t):
        angle=phase0+2*np.pi*t/sub.period_days.abs().to_numpy()
        return go.Scatter3d(x=radius*np.cos(angle),y=radius*np.sin(angle),z=np.zeros(len(sub)),
            mode='markers',text=sub.moon,customdata=sub.nominal_circular_speed_km_s,
            marker=dict(size=5,color=np.arange(len(sub)),colorscale='Turbo'),name='Moons',
            hovertemplate='%{text}<br>Circular model speed: %{customdata:.3f} km/s<extra></extra>')
    idx=len(fig.data);fig.add_trace(points(0))
    frames=[go.Frame(name=str(k),data=[points(t)],traces=[idx]) for k,t in enumerate(times)]
    style3d(fig,f'{parent} · circular teaching animation · arbitrary phases; all prograde','1000 km',max(radius)*1.1)
    animate(fig,frames,[f'{t:.3f} d' for t in times]);show(fig,'moon_animation')
    display(sub[['planet','moon','a_km','period_days','nominal_circular_speed_km_s']])
moon_demo(MOON_PLANET,MOON_DISPLAY)

,planet,moon,a_km,period_days,nominal_circular_speed_km_s
0,Jupiter,Io,421800,1.762732,17.401489
1,Jupiter,Europa,671100,3.525463,13.843223
2,Jupiter,Ganymede,1070400,7.155588,10.878447
3,Jupiter,Callisto,1882700,16.690440,8.203128


## Cell 12 · NASA/JPL geometric ephemeris client

[Astroquery Horizons documentation](https://astroquery.readthedocs.io/en/latest/jplhorizons/jplhorizons.html). Explicit Julian dates for vectors are TDB. Positions are AU and velocities AU/day; this notebook converts them. Vectors are geometric, not light-time corrected telescope positions. Cached entries preserve the queried epochs; delete the cache to force a fresh ephemeris.

In [12]:
from astroquery.jplhorizons import Horizons
Horizons.TIMEOUT=60
HORIZONS_ERRORS=[]
def horizons_vectors(target,centre,epochs):
    epochs=np.asarray(epochs,float)
    key=hashlib.sha256(json.dumps([str(target),str(centre),epochs.tolist(),'ecliptic-geometric']).encode()).hexdigest()[:24]
    file=CACHE/f'horizons_{key}.csv'
    if file.exists():return pd.read_csv(file)
    table=Horizons(id=str(target),id_type=None,location=centre,epochs=epochs.tolist()).vectors(
        refplane='ecliptic',aberrations='geometric')
    df=table.to_pandas()
    required=['datetime_jd','x','y','z','vx','vy','vz']
    if not set(required).issubset(df):raise ValueError('Unexpected Horizons response columns')
    if len(df)!=len(epochs):raise ValueError('Incomplete time coverage from Horizons')
    if not np.isfinite(df[required].to_numpy(float)).all():raise ValueError('Non-finite Horizons vectors')
    if not np.allclose(df.datetime_jd.to_numpy(),epochs,rtol=0,atol=1e-7):raise ValueError('Unexpected epoch order')
    df.to_csv(file,index=False)
    return df
print('Horizons client ready. No requests are made until RUN_HORIZONS=True.')

Horizons client ready. No requests are made until RUN_HORIZONS=True.


## Cell 13 · Optional: all eight planets from Horizons

This uses each planet’s centre, including the Earth itself. Arrows indicate instantaneous velocity direction with an arbitrary common display length; the table gives actual Sun-relative speeds.

In [13]:
if RUN_HORIZONS:
    rows=[]; fig=go.Figure()
    for name in list(PARENT_IDS)[:8]:
        try:
            d=horizons_vectors(PARENT_IDS[name],'500@10',[EPOCH_TDB])
            xyz=d[['x','y','z']].iloc[0].to_numpy(float)
            vel=d[['vx','vy','vz']].iloc[0].to_numpy(float)*AU_KM/DAY_S
            rows.append(dict(planet=name,speed_km_s=np.linalg.norm(vel),distance_AU=np.linalg.norm(xyz)))
            fig.add_trace(go.Scatter3d(x=[xyz[0]],y=[xyz[1]],z=[xyz[2]],mode='markers+text',text=[name],name=name))
            velocity_arrow(fig,xyz,vel,name+' velocity direction (arrow length arbitrary)',length=1)
        except Exception as exc:HORIZONS_ERRORS.append(dict(body=name,error=str(exc)[:250]))
    if rows:
        fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',name='Sun',marker=dict(color='gold',size=10)))
        show(style3d(fig,f'Horizons planets · Sun origin · J2000 ecliptic · {EPOCH_UTC} UTC','AU'),'horizons_planets')
        display(pd.DataFrame(rows))
    if HORIZONS_ERRORS:display(pd.DataFrame(HORIZONS_ERRORS))
else:print('Optional: set RUN_HORIZONS=True in Cell 03, then run this cell.')

Optional: set RUN_HORIZONS=True in Cell 03, then run this cell.


## Cell 14 · Optional: true 3D moon trajectories and speeds

Major moons are the practical starting point. `HORIZONS_SCOPE="all"` can require hundreds of requests for one parent and take several minutes. Requests are sequential, successful responses are cached, and missing ephemerides are reported. Change the parent and rerun to explore every system. A sampled path may not cover a full orbit. Short periods require short time steps.

In [14]:
if RUN_HORIZONS:
    sub=moons.query('planet == @HORIZONS_PARENT').copy()
    if HORIZONS_SCOPE=='major':sub=sub[sub.id.isin(MAJOR_IDS.get(HORIZONS_PARENT,[]))]
    epochs=EPOCH_TDB+np.linspace(0,HORIZONS_DAYS,HORIZONS_SAMPLES)
    tracks={}; errors=[]
    for k,(_,row) in enumerate(sub.iterrows()):
        try: tracks[row.moon]=horizons_vectors(row.id,'500@'+PARENT_IDS[HORIZONS_PARENT],epochs)
        except Exception as exc:errors.append({'moon':row.moon,'error':str(exc)[:250]})
        if (k+1)%10==0:print(f'{k+1}/{len(sub)} requested; {len(tracks)} succeeded')
    print(f'{len(tracks)}/{len(sub)} selected moons returned complete coverage.')
    if errors:display(pd.DataFrame(errors))
    if tracks:
        names=list(tracks)
        xyz=np.stack([tracks[n][['x','y','z']].to_numpy(float)*AU_KM for n in names],axis=1)
        vel=np.stack([tracks[n][['vx','vy','vz']].to_numpy(float)*AU_KM/DAY_S for n in names],axis=1)
        speed=np.linalg.norm(vel,axis=-1)
        fig=go.Figure()
        for j,n in enumerate(names):fig.add_trace(line3d(xyz[:,j,:]/1000,n+' sampled path',width=1,showlegend=False))
        fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',name=HORIZONS_PARENT,marker=dict(size=10,color='gold')))
        idx=len(fig.data)
        def real_moons(k):
            q=xyz[k]/1000
            return go.Scatter3d(x=q[:,0],y=q[:,1],z=q[:,2],mode='markers',text=names,
                customdata=speed[k],marker=dict(size=5),name='Horizons moons',
                hovertemplate='%{text}<br>Parent-relative speed: %{customdata:.3f} km/s<extra></extra>')
        fig.add_trace(real_moons(0))
        frames=[go.Frame(name=str(k),data=[real_moons(k)],traces=[idx]) for k in range(len(epochs))]
        extent=np.max(np.abs(xyz))/1000*1.1
        style3d(fig,f'{HORIZONS_PARENT} moons · Horizons · parent origin, J2000 ecliptic','1000 km',extent)
        animate(fig,frames,[f'{d:.3f} d' for d in epochs-EPOCH_TDB]);show(fig,'horizons_moons')
        display(pd.DataFrame({'moon':names,'initial_speed_km_s':speed[0],
            'minimum_sampled_speed_km_s':speed.min(axis=0),'maximum_sampled_speed_km_s':speed.max(axis=0)}))
        fastest=pd.to_numeric(sub.period_days,errors='coerce').abs().min()
        if HORIZONS_DAYS/(HORIZONS_SAMPLES-1)>fastest/20:
            print('Sampling is coarse for the fastest selected moon. Reduce HORIZONS_DAYS to avoid apparent reversed motion.')
else:print('Optional: RUN_HORIZONS=True. HORIZONS_SCOPE="all" includes all catalogue moons of your chosen parent.')

Optional: RUN_HORIZONS=True. HORIZONS_SCOPE="all" includes all catalogue moons of your chosen parent.


## Cell 15 · Milky Way model and Solar motion

The static background is a synthetic axisymmetric disc and bulge, not a reconstruction of observed spiral arms. Only the Sun marker is animated. We assume 8.2 kpc and 230 km/s to illustrate orbital direction; these are model choices. [Astropy’s Galactocentric frame documentation](https://docs.astropy.org/en/stable/coordinates/galactocentric.html) explains the coordinate conventions and why Solar parameters must be specified.

In [15]:
# Explicit, editable teaching assumptions, not fitted measurements.
SUN_RADIUS_KPC=8.2
CIRCULAR_SPEED_KM_S=230.0
rng=np.random.default_rng(42)
r=np.clip(rng.gamma(2,2.5,3500),0.2,16)
phi=rng.uniform(0,2*np.pi,len(r));z=rng.normal(0,0.18,len(r))
disc=np.c_[r*np.cos(phi),r*np.sin(phi),z]
bulge=rng.normal(0,[1.2,0.7,0.45],size=(700,3))
# Astropy-like axes: Sun lies on negative X; positive Y is the local prograde direction.
sun=np.array([-SUN_RADIUS_KPC,0,0])
period_myr=(2*np.pi*SUN_RADIUS_KPC*u.kpc/(CIRCULAR_SPEED_KM_S*u.km/u.s)).to_value(u.Myr)
fig=go.Figure()
fig.add_trace(go.Scatter3d(x=disc[:,0],y=disc[:,1],z=disc[:,2],mode='markers',name='Synthetic disc',marker=dict(size=1.5,color=r,colorscale='Blues',opacity=0.55)))
fig.add_trace(go.Scatter3d(x=bulge[:,0],y=bulge[:,1],z=bulge[:,2],mode='markers',name='Synthetic bulge',marker=dict(size=1.5,color='#edc18e',opacity=0.5)))
theta=np.linspace(0,2*np.pi,300)
fig.add_trace(line3d(np.c_[-SUN_RADIUS_KPC*np.cos(theta),SUN_RADIUS_KPC*np.sin(theta),np.zeros(len(theta))],'Assumed Solar orbit','gold',2))
fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers+text',text=['Galactic centre'],name='Centre',marker=dict(size=6,color='white')))
velocity_arrow(fig,sun,[0,1,0],f'Assumed Solar orbit velocity: {CIRCULAR_SPEED_KM_S:.0f} km/s','cyan',3)
idx=len(fig.data)
def sun_marker(t):
    a=2*np.pi*t/period_myr
    return go.Scatter3d(x=[-SUN_RADIUS_KPC*np.cos(a)],y=[SUN_RADIUS_KPC*np.sin(a)],z=[0],mode='markers+text',text=['Sun'],name='Sun',marker=dict(size=7,color='gold'))
fig.add_trace(sun_marker(0))
times=np.linspace(0,period_myr,101)
frames=[go.Frame(name=str(k),data=[sun_marker(t)],traces=[idx]) for k,t in enumerate(times)]
style3d(fig,'Milky Way teaching model · synthetic disc; Sun on an assumed circular orbit','kpc',17)
animate(fig,frames,[f'{t:.0f} Myr' for t in times]);show(fig,'milky_way')
print(f'Assumed radius {SUN_RADIUS_KPC} kpc; speed {CIRCULAR_SPEED_KM_S} km/s; orbital period {period_myr:.1f} million years.')

Assumed radius 8.2 kpc; speed 230.0 km/s; orbital period 219.0 million years.


## Cell 16 · Direction of the Galactic centre in your actual sky

We live **inside** the Milky Way. “Which direction is the Milky Way?” can mean its centre or its band across the sky; this plot shows both. Enter UTC, not Melbourne local time. Earth-orientation predictions bundled with Astropy are adequate for this educational view, not precision telescope pointing.

In [16]:
observer=EarthLocation(lat=LATITUDE_DEG*u.deg,lon=LONGITUDE_DEG*u.deg,height=HEIGHT_M*u.m)
obstime=Time(EPOCH_UTC,scale='utc')
frame=AltAz(obstime=obstime,location=observer,pressure=0*u.hPa)
gc=SkyCoord(l=0*u.deg,b=0*u.deg,frame='galactic')
gc_altaz=gc.transform_to(frame)
plane=SkyCoord(l=np.linspace(0,360,721)*u.deg,b=np.zeros(721)*u.deg,frame='galactic').transform_to(frame)
# Local Cartesian: X east, Y north, Z zenith; a unit sphere shows direction, not distance.
def sky_xyz(alt,az):
    alt=np.deg2rad(alt);az=np.deg2rad(az)
    return np.c_[np.cos(alt)*np.sin(az),np.cos(alt)*np.cos(az),np.sin(alt)]
q=sky_xyz(plane.alt.deg,plane.az.deg)
fig=go.Figure()
for mask,name,color in [(plane.alt.deg>=0,'Galactic plane above horizon','#9de3ff'),(plane.alt.deg<0,'Galactic plane below horizon','#495264')]:
    a=q.copy();a[~mask]=np.nan;fig.add_trace(line3d(a,name,color,4))
th=np.linspace(0,2*np.pi,200)
fig.add_trace(line3d(np.c_[np.sin(th),np.cos(th),np.zeros(len(th))],'Horizon','gold',3))
point=sky_xyz([gc_altaz.alt.deg],[gc_altaz.az.deg])[0]
fig.add_trace(go.Scatter3d(x=[point[0]],y=[point[1]],z=[point[2]],mode='markers+text',text=['Galactic centre'],marker=dict(size=7,color='red'),name='Galactic centre'))
for label,p in [('N',[0,1,0]),('E',[1,0,0]),('S',[0,-1,0]),('W',[-1,0,0]),('Zenith',[0,0,1])]:
    fig.add_trace(go.Scatter3d(x=[p[0]],y=[p[1]],z=[p[2]],mode='text',text=[label],showlegend=False))
style3d(fig,f'Your sky · lat {LATITUDE_DEG}, lon {LONGITUDE_DEG} · {EPOCH_UTC} UTC','unit direction',1.1)
fig.update_layout(scene=dict(xaxis_title='East',yaxis_title='North',zaxis_title='Up'))
show(fig,'local_sky')
print(f'Galactic centre: altitude {gc_altaz.alt.deg:.1f}°, azimuth {gc_altaz.az.deg:.1f}° (north=0°, east=90°).')
print('Above horizon' if gc_altaz.alt.deg>0 else 'Below horizon')
print('Geometric sky directions only: visibility also depends on daylight, weather, Moon and light pollution.')

Galactic centre: altitude 49.7°, azimuth 268.1° (north=0°, east=90°).
Above horizon
Geometric sky directions only: visibility also depends on daylight, weather, Moon and light pollution.


## Cell 17 · Optional: download a measured nearby-star sample

Source: [Gaia TAP through Astroquery](https://astroquery.readthedocs.io/en/latest/gaia/gaia.html). This selects the nearest stars meeting the filters up to the row limit. A radial-velocity requirement removes many stars. Inverse parallax is used only after a high signal-to-noise cut; parallax systematics and selection biases remain.

In [17]:
stars=None
if RUN_GAIA:
    from astroquery.gaia import Gaia
    # Quality-selected sample with all 6 phase-space coordinates, not volume-complete.
    query=f"""SELECT TOP {int(GAIA_MAX_STARS)} source_id,ra,dec,parallax,
        pmra,pmdec,radial_velocity,phot_g_mean_mag,ref_epoch
        FROM gaiadr3.gaia_source
        WHERE parallax > {1000/GAIA_RADIUS_PC:.8f}
          AND parallax_over_error > 20 AND ruwe < 1.4
          AND radial_velocity IS NOT NULL AND pmra IS NOT NULL AND pmdec IS NOT NULL
        ORDER BY parallax DESC"""
    key=hashlib.sha256(query.encode()).hexdigest()[:16];file=CACHE/f'gaia_{key}.csv'
    try:
        stars=pd.read_csv(file,dtype={'source_id':str}) if file.exists() else Gaia.launch_job_async(query,dump_to_file=False).get_results().to_pandas()
        stars['source_id']=stars.source_id.astype(str)
        stars.to_csv(file,index=False)
        print(len(stars),'quality-selected Gaia DR3 stars with 3D velocities; epoch J2016.0')
        display(stars.head())
    except Exception as exc:print('Gaia unavailable; no fabricated replacement:',str(exc)[:250])
else:print('Optional: set RUN_GAIA=True and rerun this cell and the next two.')

Optional: set RUN_GAIA=True and rerun this cell and the next two.


## Cell 18 · Optional: measured stellar speed and direction in 3D

Galactic axes here have X toward the Galactic centre, Y toward longitude 90°, and Z toward Galactic north, with the **Sun at the origin**. The scalar speed combines transverse and radial motion. Gaia `pmra` already includes cos(dec). It must not be multiplied by cos(dec) again.

In [18]:
star_position=None;star_velocity=None
if stars is not None and len(stars):
    c=SkyCoord(ra=stars.ra.to_numpy()*u.deg,dec=stars.dec.to_numpy()*u.deg,
        distance=(1000/stars.parallax.to_numpy())*u.pc,
        pm_ra_cosdec=stars.pmra.to_numpy()*u.mas/u.yr,pm_dec=stars.pmdec.to_numpy()*u.mas/u.yr,
        radial_velocity=stars.radial_velocity.to_numpy()*u.km/u.s,
        obstime=Time(stars.ref_epoch.to_numpy(),format='jyear',scale='tcb'))
    g=c.galactic
    star_position=g.cartesian.xyz.to_value(u.pc).T
    star_velocity=g.velocity.d_xyz.to_value(u.km/u.s).T
    speeds=np.linalg.norm(star_velocity,axis=1)
    fig=go.Figure(go.Scatter3d(x=star_position[:,0],y=star_position[:,1],z=star_position[:,2],
        mode='markers',text=stars.source_id,
        customdata=speeds,marker=dict(size=3,color=speeds,colorscale='Turbo',colorbar=dict(title='km/s')),
        hovertemplate='Gaia %{text}<br>Sun-relative speed %{customdata:.2f} km/s<extra></extra>',name='Gaia stars'))
    # Each arrow is displacement over 10,000 years, so lengths encode speed.
    tip=star_position+star_velocity*KM_S_TO_PC_YR*10000
    for j in range(min(60,len(stars))):fig.add_trace(line3d(np.array([star_position[j],tip[j]]),'10,000-year velocity guide',color='#8ddff4',width=2,showlegend=False))
    fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',name='Solar origin',marker=dict(size=8,color='gold')))
    show(style3d(fig,'Measured Gaia DR3 sample · Galactic axes, Solar origin · epoch J2016.0','pc'),'gaia_stars')
    stars['speed_relative_to_Sun_km_s']=speeds
    display(stars[['source_id','radial_velocity','speed_relative_to_Sun_km_s']].head(15))
else:print('Run the Gaia download cell with RUN_GAIA=True first.')

Run the Gaia download cell with RUN_GAIA=True first.


## Cell 19 · Optional: animate the measured stars

This is $\mathbf{x}(t)=\mathbf{x}_0+\mathbf{v}t$. It neglects acceleration, encounters and binary orbital effects. Do not interpret it as a reliable long-term prediction or a simulation of stars orbiting the Galactic centre.

In [19]:
if star_position is not None:
    times=np.linspace(0,STAR_YEARS,61)
    fig=go.Figure()
    def stellar_marker(t):
        q=star_position+star_velocity*KM_S_TO_PC_YR*t
        return go.Scatter3d(x=q[:,0],y=q[:,1],z=q[:,2],mode='markers',text=stars.source_id,
            marker=dict(size=3,color=np.linalg.norm(star_velocity,axis=1),colorscale='Turbo'),
            name='Straight-line extrapolation',hovertemplate='Gaia %{text}<extra></extra>')
    fig.add_trace(stellar_marker(0))
    fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',name='Solar origin',marker=dict(size=8,color='gold')))
    frames=[go.Frame(name=str(k),data=[stellar_marker(t)],traces=[0]) for k,t in enumerate(times)]
    final=star_position+star_velocity*KM_S_TO_PC_YR*STAR_YEARS
    extent=max(np.max(np.abs(star_position)),np.max(np.abs(final)))*1.1
    style3d(fig,'Gaia velocities · linear extrapolation after J2016.0, no gravitational integration','pc',extent)
    animate(fig,frames,[f'{t:.0f} yr' for t in times]);show(fig,'gaia_motion')
else:print('Waiting for optional Gaia data.')

Waiting for optional Gaia data.


## Cell 20 · Optional: known exoplanet hosts across the Milky Way

[NASA Exoplanet Archive TAP](https://exoplanetarchive.ipac.caltech.edu/docs/TAP/usingTAP.html). There is no TOP limit in this query. Multiple planets around a star share the same host position at this scale, so one marker represents one host. These are discovery-biased known systems, not all planets in the Milky Way. Host proper motions are not downloaded, so this map is static.

In [20]:
exoplanets=None
if RUN_EXOPLANETS:
    query="""SELECT pl_name,hostname,ra,dec,sy_dist,pl_orbper,pl_orbsmax,st_mass
        FROM pscomppars WHERE sy_dist > 0 AND ra IS NOT NULL AND dec IS NOT NULL"""
    key=hashlib.sha256(query.encode()).hexdigest()[:16];file=CACHE/f'exoplanets_{key}.csv'
    try:
        if file.exists():exoplanets=pd.read_csv(file)
        else:
            response=requests.get('https://exoplanetarchive.ipac.caltech.edu/TAP/sync',
                params={'query':query,'format':'csv'},timeout=120)
            response.raise_for_status();exoplanets=pd.read_csv(io.StringIO(response.text))
            if not {'pl_name','hostname','ra','dec','sy_dist'}.issubset(exoplanets):raise ValueError('Unexpected archive schema')
            exoplanets.to_csv(file,index=False)
        print(len(exoplanets),'archive planet rows with valid host coordinates and distance, before numeric checks.')
        for col in ['ra','dec','sy_dist','pl_orbper','pl_orbsmax','st_mass']:
            exoplanets[col]=pd.to_numeric(exoplanets[col],errors='coerce')
        exoplanets=exoplanets.dropna(subset=['ra','dec','sy_dist'])
        exoplanets=exoplanets[exoplanets.sy_dist>0].copy()
        hosts=exoplanets.groupby('hostname').agg(ra=('ra','first'),dec=('dec','first'),distance_pc=('sy_dist','first'),known_planets=('pl_name','size')).reset_index()
        coords=SkyCoord(ra=hosts.ra.to_numpy()*u.deg,dec=hosts.dec.to_numpy()*u.deg,distance=hosts.distance_pc.to_numpy()*u.pc).galactic.cartesian.xyz.to_value(u.pc).T
        fig=go.Figure(go.Scatter3d(x=coords[:,0],y=coords[:,1],z=coords[:,2],mode='markers',text=hosts.hostname,
            customdata=hosts[['known_planets','distance_pc']].to_numpy(),
            marker=dict(size=3,color=hosts.known_planets,colorscale='Viridis',colorbar=dict(title='Planets')),
            hovertemplate='%{text}<br>%{customdata[0]} planets<br>%{customdata[1]:.1f} pc<extra></extra>',name='Known hosts'))
        fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',name='Sun',marker=dict(size=8,color='gold')))
        show(style3d(fig,'NASA archive exoplanet host systems · Galactic axes, Sun origin','pc'),'exoplanet_hosts')
        display(hosts.sort_values('known_planets',ascending=False).head(15))
    except Exception as exc:print('Exoplanet archive unavailable:',str(exc)[:250])
else:print('Optional: set RUN_EXOPLANETS=True and rerun.')

Optional: set RUN_EXOPLANETS=True and rerun.


## Cell 21 · Optional: inspect any downloaded exoplanet system

This provides orbit sizes and circular speed estimates, not true orbital inclinations, eccentricities, phases or moon systems. The missing semimajor-axis estimate uses $a^3\simeq M_\star P^2$ with AU, solar masses and years. We do not invent exomoons.

In [21]:
EXOPLANET_HOST='TRAPPIST-1'  # edit to any hostname in the downloaded table
if exoplanets is not None:
    system=exoplanets[exoplanets.hostname.str.casefold()==EXOPLANET_HOST.casefold()].copy()
    if system.empty:print('No matching host in the downloaded valid-distance subset.')
    else:
        # Fill missing semimajor axes only when period and stellar mass exist; planet mass ignored.
        estimated=system.pl_orbsmax.isna() & system.pl_orbper.gt(0) & system.st_mass.gt(0)
        system.loc[estimated,'pl_orbsmax']=(system.loc[estimated,'st_mass']*(system.loc[estimated,'pl_orbper']/365.25)**2)**(1/3)
        system['axis_source']=np.where(estimated,'Kepler estimate','archive')
        system=system[system.pl_orbsmax.gt(0)&system.pl_orbper.gt(0)].copy()
        if not system.empty:
            system['circular_speed_km_s']=2*np.pi*system.pl_orbsmax*AU_KM/(system.pl_orbper*DAY_S)
            fig=go.Figure();a=np.linspace(0,2*np.pi,200)
            for _,r in system.iterrows():fig.add_trace(line3d(np.c_[r.pl_orbsmax*np.cos(a),r.pl_orbsmax*np.sin(a),np.zeros(len(a))],r.pl_name,width=3))
            fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',name=EXOPLANET_HOST,marker=dict(size=9,color='gold')))
            show(style3d(fig,EXOPLANET_HOST+' · schematic circular coplanar orbit sizes, not measured orientations','AU'),'exoplanet_system')
            display(system[['pl_name','pl_orbper','pl_orbsmax','axis_source','circular_speed_km_s']])
        else:print('This host lacks usable orbital periods / semimajor axes in the selected table.')
else:print('Download the optional exoplanet table first.')

Download the optional exoplanet table first.


## Cell 22 · Nearby galaxies: real locations, rounded catalogue values

Rounded values follow the nearby-galaxy compilation of [McConnachie (2012)](https://arxiv.org/abs/1204.1562). This map shows four selected systems, not a complete galaxy catalogue. The Milky Way centre is not exactly the Sun’s location. The arrows visualize only the measured line-of-sight component; unknown tangential components are not set to zero as a physical claim. Do not use these arrows to predict a Milky Way–Andromeda collision.

In [22]:
# Rounded illustrative reference values; positions in ICRS degrees, distances in kpc,
# systemic heliocentric radial velocities in km/s. Not a precision dynamical dataset.
galaxies=pd.DataFrame([
    ['Large Magellanic Cloud',80.8942,-69.7561,50.0,262.2],
    ['Small Magellanic Cloud',13.1867,-72.8286,61.0,145.6],
    ['Andromeda (M31)',10.6847,41.2688,783.0,-300.0],
    ['Triangulum (M33)',23.4621,30.6602,809.0,-179.2]],
    columns=['galaxy','ra_deg','dec_deg','distance_kpc','heliocentric_radial_km_s'])
galcoords=SkyCoord(ra=galaxies.ra_deg.to_numpy()*u.deg,dec=galaxies.dec_deg.to_numpy()*u.deg,
    distance=galaxies.distance_kpc.to_numpy()*u.kpc).galactic
galxyz=galcoords.cartesian.xyz.to_value(u.kpc).T
fig=go.Figure(go.Scatter3d(x=galxyz[:,0],y=galxyz[:,1],z=galxyz[:,2],mode='markers+text',text=galaxies.galaxy,
    customdata=galaxies[['distance_kpc','heliocentric_radial_km_s']].to_numpy(),marker=dict(size=7,color='#8fcaff'),
    hovertemplate='%{text}<br>Distance %{customdata[0]:.0f} kpc<br>Heliocentric radial velocity %{customdata[1]:.1f} km/s<extra></extra>',name='Galaxies'))
fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers+text',text=['Solar location in Milky Way'],marker=dict(size=8,color='gold'),name='Solar origin'))
for j,row in galaxies.iterrows():
    radial_vector=galxyz[j]/np.linalg.norm(galxyz[j])*row.heliocentric_radial_km_s
    velocity_arrow(fig,galxyz[j],radial_vector,row.galaxy+' radial component ONLY',length=abs(row.heliocentric_radial_km_s)*0.25)
show(style3d(fig,'Local Group neighbourhood · radial velocity components ONLY · Sun origin','kpc'),'nearby_galaxies')
display(galaxies)
print('Negative radial velocity means approaching the Sun; it is NOT the full 3D speed.')

,galaxy,ra_deg,dec_deg,distance_kpc,heliocentric_radial_km_s
0,Large Magellanic Cloud,80.8942,-69.7561,50.0,262.2
1,Small Magellanic Cloud,13.1867,-72.8286,61.0,145.6
2,Andromeda (M31),10.6847,41.2688,783.0,-300.0
3,Triangulum (M33),23.4621,30.6602,809.0,-179.2


Negative radial velocity means approaching the Sun; it is NOT the full 3D speed.


## Cell 23 · Expansion of the Universe: a separate teaching model

This is a first-order expansion model, not a numerical ΛCDM simulation and not an observed cosmic web. The origin is an arbitrary observer, not the centre of the Universe. Gravitationally bound systems such as the Solar System and Milky Way do not expand this way. The 250 markers represent synthetic unbound tracers.

In [23]:
H0=70.0  # assumed km/s/Mpc, editable; not a new measurement
rng=np.random.default_rng(15)
pos=rng.uniform(-70,70,(250,3))
distance=np.linalg.norm(pos,axis=1)
recession_speed=H0*distance
# First-order expansion: a(t)=1+H0*t; useful over short fractions of a Hubble time.
H0_per_Gyr=(H0*u.km/u.s/u.Mpc).to_value(1/u.Gyr)
times=np.linspace(0,1,61)
fig=go.Figure()
def expansion_marker(t):
    q=pos*(1+H0_per_Gyr*t)
    return go.Scatter3d(x=q[:,0],y=q[:,1],z=q[:,2],mode='markers',
        customdata=np.c_[distance,recession_speed],
        marker=dict(size=3,color=recession_speed,colorscale='Plasma',cmin=0,cmax=recession_speed.max(),colorbar=dict(title='Initial km/s')),
        hovertemplate='Synthetic point<br>Initial distance %{customdata[0]:.1f} Mpc<br>Initial Hubble recession %{customdata[1]:.0f} km/s<extra></extra>',name='Synthetic unbound tracers')
fig.add_trace(expansion_marker(0))
frames=[go.Frame(name=str(k),data=[expansion_marker(t)],traces=[0]) for k,t in enumerate(times)]
style3d(fig,'Hubble-law teaching model · synthetic tracers · initial H₀ = 70 km/s/Mpc','Mpc',85)
animate(fig,frames,[f'{t:.2f} Gyr' for t in times]);show(fig,'cosmic_expansion')
print('Recession speed v = H0 × distance. This is not local orbital velocity through space.')

Recession speed v = H0 × distance. This is not local orbital velocity through space.


## Cell 24 · Sanity checks and the meaning of each speed

Never add speeds from different frames as scalar numbers. Velocity is a vector: a Moon’s Sun-relative velocity needs its planet’s velocity added **in matching axes and at the same epoch**. This notebook deliberately keeps those frames separate.

In [24]:
assert planet_xyz(EPOCH_TDB).shape==(1,8,3)
assert np.isfinite(planet_xyz(EPOCH_TDB)).all()
assert (np.linalg.norm(v0,axis=1)>0).all()
assert 25 < np.linalg.norm(v0[2]) < 35  # Earth–Moon barycentre near 30 km/s
assert 4 < np.linalg.norm(v0[7]) < 7    # Neptune near 5 km/s
assert moons.id.is_unique
assert moons.a_km.gt(0).all()
assert moons.period_days.abs().gt(0).all()
assert 180 < period_myr < 300
frames_table=pd.DataFrame([
    ['Planet','Sun','Approximate JPL model or optional Horizons','km/s'],
    ['Moon','Its parent planet','Circular estimate OR Horizons, explicitly labelled','km/s'],
    ['Solar Galactic orbit','Galactic centre','230 km/s is an assumed circular model','km/s'],
    ['Gaia star','Sun / Solar origin','Measured transverse + radial components','km/s'],
    ['Nearby galaxy','Sun','Systemic radial component only, not 3D speed','km/s'],
    ['Expansion tracer','Chosen comoving observer','Assumed Hubble recession, synthetic data','km/s']],
    columns=['object','reference','interpretation','unit'])
display(frames_table)
print('Basic numerical checks passed. They do not validate the optional remote services.')

,object,reference,interpretation,unit
0,Planet,Sun,Approximate JPL model or optional Horizons,km/s
1,Moon,Its parent planet,"Circular estimate OR Horizons, explicitly labe...",km/s
2,Solar Galactic orbit,Galactic centre,230 km/s is an assumed circular model,km/s
3,Gaia star,Sun / Solar origin,Measured transverse + radial components,km/s
4,Nearby galaxy,Sun,"Systemic radial component only, not 3D speed",km/s
5,Expansion tracer,Chosen comoving observer,"Assumed Hubble recession, synthetic data",km/s


Basic numerical checks passed. They do not validate the optional remote services.


## Cell 25 · Optional: save every plot as a standalone HTML file

These exports retain rotation and frame sliders; no Jupyter server is required to reopen an HTML plot. Each HTML embeds Plotly, so files can be several MB. Plotly 3D animation redraws frames rather than interpolating a physical trajectory between them.

In [25]:
if EXPORT_HTML:
    out=Path('universe_exports');out.mkdir(exist_ok=True)
    for name,fig in FIGURES.items():
        fig.write_html(out/f'{name}.html',include_plotlyjs=True,full_html=True,auto_play=False)
    speed_table.to_csv(out/'planet_speeds_approximate.csv',index=False)
    moons.to_csv(out/'jpl_moon_catalogue.csv',index=False)
    galaxies.to_csv(out/'selected_nearby_galaxies.csv',index=False)
    if stars is not None:stars.to_csv(out/'gaia_sample.csv',index=False)
    if exoplanets is not None:exoplanets.to_csv(out/'exoplanets.csv',index=False)
    print(f'Wrote {len(FIGURES)} self-contained HTML plots and data tables to {out.resolve()}')
else:print('Set EXPORT_HTML=True in Cell 03 and rerun this cell to save rotatable plots for your browser.')

Set EXPORT_HTML=True in Cell 03 and rerun this cell to save rotatable plots for your browser.


## Troubleshooting

- **No plot / blank plot:** Trust the notebook, rerun Cell 02 and the plot cell. Try `pio.renderers.default = 'browser'` for a local desktop Jupyter server, or use the HTML export cell. Browser rendering is unsuitable for a remote server without a desktop.
- **ModuleNotFoundError:** rerun Cell 01, restart the kernel, then start at Cell 02. Use the same kernel for installation and execution.
- **A plot is slow:** use the major moon subset, lower `GAIA_MAX_STARS`, and close other large plot outputs. Hundreds of orbit traces can be demanding for a browser.
- **NASA/ESA request fails:** the offline sections still run. Retry only the affected cell later. The error text is printed; unavailable objects are not fabricated.
- **NameError after skipping cells:** use Run All after the installation cell, or run the dependency cells in order.
- **Moon appears to reverse:** time samples can alias a fast orbit. Reduce the animation duration or increase samples. The offline schematic deliberately assigns prograde motion to every moon and must not be used to infer real directions.
- **Why don’t I see every planet in the Milky Way?** Most are undiscovered; the archive map covers only known systems with valid distances. Moons of all exoplanets are not known.
- **Why no one single zoomable Universe with everything?** AU, pc, kpc and Mpc differ enormously. Separate views preserve readable spatial scales and valid reference frames.

## Sources and provenance

Accessed for notebook preparation on 10 September 2026. Live catalogue results depend on your later execution date; cached data persist until you remove the cache. Embedded moon values are a dated snapshot, not a live discovery count.

1. [JPL approximate planetary positions and 1800–2050 elements](https://ssd.jpl.nasa.gov/planets/approx_pos.html).
2. [JPL planetary satellite mean elements](https://ssd.jpl.nasa.gov/sats/elem/) — embedded catalogue; JPL warns against using mean elements as precision ephemerides.
3. [Astroquery Horizons vectors, coordinate origins and units](https://astroquery.readthedocs.io/en/latest/jplhorizons/jplhorizons.html).
4. [Astropy Galactocentric frame conventions](https://docs.astropy.org/en/stable/coordinates/galactocentric.html).
5. [Gaia TAP queries](https://astroquery.readthedocs.io/en/latest/gaia/gaia.html), [Gaia DR3 documentation](https://gea.esac.esa.int/archive/documentation/GDR3/).
6. [NASA Exoplanet Archive TAP access](https://exoplanetarchive.ipac.caltech.edu/docs/TAP/usingTAP.html), [planetary-system composite table definitions](https://exoplanetarchive.ipac.caltech.edu/docs/API_PSCompPars_columns.html).
7. [McConnachie 2012, observed properties of dwarf galaxies in and around the Local Group](https://arxiv.org/abs/1204.1562) — rounded selected-galaxy values.
8. [NASA: the expanding Universe](https://science.nasa.gov/universe/).

**Interpretation:** the default notebook is a reproducible educational explorer. Optional Horizons and Gaia sections are grounded in measured/modelled astronomical data; their coverage is explicitly reported. It is not a precision navigation tool, a complete Universe catalogue or a gravitational N-body simulation.

## Validation performed during preparation

- All 25 default code cells executed successfully in an in-process IPython session after package installation; numerical sanity checks passed. A separate Jupyter kernel could not be launched in the preparation environment, so desktop Jupyter rendering was not visually verified here.
- The optional Horizons moon path was tested against the Earth–Moon system with five epochs and returned complete vectors.
- The NASA exoplanet query returned 6,332 rows with host coordinates and distances during testing; the host map and selected-system code executed. Your later download may differ.
- The live Gaia service was unreachable from the preparation environment. Its coordinate conversion and animation were tested using a two-row artificial fixture, which is **not** included as astronomical data.
- The saved default outputs are educational model plots. For measured Gaia stars or geometric JPL positions, enable the corresponding controls and run those sections.
